In [ ]:
from pathlib import Path
from snowflake.snowpark.context import get_active_session
from datetime import datetime, timezone
from decimal import Decimal
import json
import re

session = get_active_session()
EXPECTED_ACCOUNT = '<your account>'
DATABASE = 'YOUR_DATABASE'  # existing approved standard database, not USER$...
SCHEMA = '<your schema>'
WAREHOUSE = None  # None uses your current active warehouse; or enter its name
APPROVE_SETUP = False
APPROVE_CREATE_MCP = False
query_evidence = []
setup_attempted = False
checks_verified = False

def identifier(value):
    if not isinstance(value, str) or not re.fullmatch(r'[A-Za-z_][A-Za-z0-9_]*', value):
        raise ValueError('Use a simple unquoted identifier')
    return value.upper()

def context():
    return session.sql('SELECT CURRENT_ACCOUNT_NAME() AS ACCOUNT_NAME, CURRENT_ROLE() AS ROLE_NAME, CURRENT_DATABASE() AS DATABASE_NAME, CURRENT_WAREHOUSE() AS WAREHOUSE_NAME').collect()[0].as_dict()

def checked_context():
    actual = context()
    if actual['ACCOUNT_NAME'] != EXPECTED_ACCOUNT:
        raise RuntimeError(f"Expected {EXPECTED_ACCOUNT}; active account is {actual['ACCOUNT_NAME']}")
    if DATABASE == 'YOUR_DATABASE' or DATABASE.upper() in {'SNOWFLAKE'}:
        raise ValueError('Set DATABASE to an approved existing standard database')
    identifier(DATABASE)
    identifier(SCHEMA)
    return actual

def execute(statement, label):
    checked_context()
    event = {'label': label, 'utc': datetime.now(timezone.utc).isoformat()}
    history = None
    try:
        with session.query_history() as history:
            rows = session.sql(statement).collect()
        event.update(status='ok', rows=[row.as_dict() for row in rows])
        return rows
    except Exception as exc:
        event.update(status='error', error_type=type(exc).__name__)
        raise
    finally:
        event['query_ids'] = [query.query_id for query in history.queries] if history is not None else []
        query_evidence.append(event)

print(context())  # read-only; inspect before approving any changes

In [ ]:
from snowflake.snowpark.context import get_active_session
from datetime import datetime, timezone
from decimal import Decimal
import json
import re

session = get_active_session()
EXPECTED_ACCOUNT = '<your account>'
DATABASE = '<your db>'  # created by the approved setup cell if absent
SCHEMA = '<your schema>'
ROLE = '<your role>'
WAREHOUSE = '<your warehouse>'
APPROVE_SETUP = False
APPROVE_CREATE_MCP = False
query_evidence = []
setup_attempted = False
checks_verified = False

def identifier(value):
    if not isinstance(value, str) or not re.fullmatch(r'[A-Za-z_][A-Za-z0-9_]*', value):
        raise ValueError('Use a simple unquoted identifier')
    return value.upper()

def context():
    return session.sql('SELECT CURRENT_ACCOUNT_NAME() AS ACCOUNT_NAME, CURRENT_ROLE() AS ROLE_NAME, CURRENT_DATABASE() AS DATABASE_NAME, CURRENT_WAREHOUSE() AS WAREHOUSE_NAME').collect()[0].as_dict()

def checked_context():
    actual = context()
    if actual['ACCOUNT_NAME'] != EXPECTED_ACCOUNT:
        raise RuntimeError(f"Expected {EXPECTED_ACCOUNT}; active account is {actual['ACCOUNT_NAME']}")
    if DATABASE == 'YOUR_DATABASE' or DATABASE.upper() in {'SNOWFLAKE'}:
        raise ValueError('Set DATABASE to an approved existing standard database')
    identifier(DATABASE)
    identifier(SCHEMA)
    return actual

def execute(statement, label):
    checked_context()
    event = {'label': label, 'utc': datetime.now(timezone.utc).isoformat()}
    history = None
    try:
        with session.query_history() as history:
            rows = session.sql(statement).collect()
        event.update(status='ok', rows=[row.as_dict() for row in rows])
        return rows
    except Exception as exc:
        event.update(status='error', error_type=type(exc).__name__)
        raise
    finally:
        event['query_ids'] = [query.query_id for query in history.queries] if history is not None else []
        query_evidence.append(event)

print(context())  # read-only; inspect before approving any changes

In [ ]:
actual = checked_context()
namespace = f'{identifier(DATABASE)}.{identifier(SCHEMA)}'
role = identifier(ROLE)
warehouse = identifier(WAREHOUSE)
execute(f'USE ROLE {role}', 'select_role')
execute(f'USE WAREHOUSE {warehouse}', 'select_compute')
actual = checked_context()
if actual['ROLE_NAME'] != role or actual['WAREHOUSE_NAME'] != warehouse:
    raise RuntimeError('Active role or warehouse does not match the configured demo context')
print({'target': namespace, 'role': role, 'warehouse': warehouse, 'account': actual['ACCOUNT_NAME']})

setup_statements = [
    f'CREATE DATABASE IF NOT EXISTS {identifier(DATABASE)}',
    f"CREATE SCHEMA IF NOT EXISTS {namespace} COMMENT = 'Isolated synthetic verification demo; no production data'",
    f'''CREATE OR REPLACE TABLE {namespace}.WATERFALL_CASES (
        CASE_ID VARCHAR NOT NULL, PROCEEDS NUMBER(18,2) NOT NULL,
        EXPECTED_COUNT NUMBER(10,0) NOT NULL, TOLERANCE NUMBER(18,2) NOT NULL,
        CURRENCY VARCHAR NOT NULL)''',
    f'''INSERT INTO {namespace}.WATERFALL_CASES VALUES
        ('BALANCED',300000000,2,0.01,'USD'), ('SHORTFALL',300000000,2,0.01,'USD'),
        ('OVERALLOCATED',300000000,2,0.01,'USD'), ('MISSING',300000000,2,0.01,'USD'),
        ('INCOMPLETE',300000000,2,0.01,'USD')''',
    f'''CREATE OR REPLACE TABLE {namespace}.WATERFALL_ALLOCATIONS (
        CASE_ID VARCHAR NOT NULL, ALLOCATION_ID VARCHAR NOT NULL, AMOUNT NUMBER(18,2))''',
    f'''INSERT INTO {namespace}.WATERFALL_ALLOCATIONS VALUES
        ('BALANCED','B1',200000000),('BALANCED','B2',100000000),
        ('SHORTFALL','S1',200000000),('SHORTFALL','S2',37000000),
        ('OVERALLOCATED','O1',200000000),('OVERALLOCATED','O2',101000000),
        ('INCOMPLETE','I1',200000000),('INCOMPLETE','I2',NULL)''',
    f'''CREATE OR REPLACE VIEW {namespace}.WATERFALL_RECONCILIATION AS
    WITH allocation_summary AS (
      SELECT CASE_ID, COUNT(*) AS ACTUAL_COUNT,
        COUNT(*) - COUNT(AMOUNT) AS NULL_AMOUNT_COUNT,
        COUNT(*) - COUNT(DISTINCT ALLOCATION_ID) AS DUPLICATE_COUNT,
        SUM(IFF(AMOUNT < 0, 1, 0)) AS NEGATIVE_COUNT, SUM(AMOUNT) AS OBSERVED_TOTAL
      FROM {namespace}.WATERFALL_ALLOCATIONS GROUP BY CASE_ID
    ), checked AS (
      SELECT cases.*, COALESCE(allocations.ACTUAL_COUNT,0) AS ACTUAL_COUNT,
        COALESCE(allocations.NULL_AMOUNT_COUNT,0) AS NULL_AMOUNT_COUNT,
        COALESCE(allocations.DUPLICATE_COUNT,0) AS DUPLICATE_COUNT,
        COALESCE(allocations.NEGATIVE_COUNT,0) AS NEGATIVE_COUNT, allocations.OBSERVED_TOTAL,
        (cases.EXPECTED_COUNT > 0 AND cases.PROCEEDS >= 0 AND cases.TOLERANCE >= 0
         AND cases.CURRENCY = 'USD' AND COUNT(*) OVER (PARTITION BY cases.CASE_ID) = 1
         AND COALESCE(allocations.ACTUAL_COUNT,0) = cases.EXPECTED_COUNT
         AND COALESCE(allocations.NULL_AMOUNT_COUNT,0) = 0
         AND COALESCE(allocations.DUPLICATE_COUNT,0) = 0
         AND COALESCE(allocations.NEGATIVE_COUNT,0) = 0) AS COMPLETE
      FROM {namespace}.WATERFALL_CASES cases
      LEFT JOIN allocation_summary allocations ON cases.CASE_ID = allocations.CASE_ID
    )
    SELECT *, IFF(COMPLETE,OBSERVED_TOTAL,NULL) AS ALLOCATED_TOTAL,
      IFF(COMPLETE,PROCEEDS-OBSERVED_TOTAL,NULL) AS GAP,
      CASE WHEN NOT COMPLETE THEN 'blocked_incomplete'
           WHEN ABS(PROCEEDS-OBSERVED_TOTAL)>TOLERANCE THEN 'blocked_reconciliation'
           ELSE 'checks_passed' END AS STATUS FROM checked''',
    f'''CREATE OR REPLACE FUNCTION {namespace}.REVIEW_RECONCILIATION(P_CASE_ID VARCHAR)
    RETURNS OBJECT LANGUAGE SQL AS $$
      SELECT IFF(COUNT(*)=1,
        ANY_VALUE(OBJECT_CONSTRUCT_KEEP_NULL(
          'case_id',results.CASE_ID,'currency',results.CURRENCY,
          'proceeds',results.PROCEEDS,'allocated_total',results.ALLOCATED_TOTAL,
          'gap',results.GAP,'tolerance',results.TOLERANCE,
          'expected_count',results.EXPECTED_COUNT,'actual_count',results.ACTUAL_COUNT,
          'null_amount_count',results.NULL_AMOUNT_COUNT,'duplicate_count',results.DUPLICATE_COUNT,
          'negative_count',results.NEGATIVE_COUNT,'status',results.STATUS,
          'source','WATERFALL_RECONCILIATION','fixture_version','v1',
          'synthetic',TRUE,'payment_authorized',FALSE)),
        OBJECT_CONSTRUCT('status',IFF(COUNT(*)=0,'blocked_unknown_case','blocked_nonunique_case'),
                         'synthetic',TRUE,'payment_authorized',FALSE))
      FROM {namespace}.WATERFALL_RECONCILIATION results WHERE results.CASE_ID=P_CASE_ID
    $$''',
]
for statement in setup_statements:
    print(statement + ';\n')

In [ ]:
%%sql -r dataframe_1
show databases;

In [ ]:
APPROVE_SETUP=True
setup_attempted=False
APPROVE_SKILL_UPLOAD_AND_AGENT=True
APPROVE_ONE_BILLABLE_EVALUATION=True
APPROVE_SEC_DEPLOY=True
APPROVE_TRULENS_RUN=True
APPROVE_JUDGE_METRICS=True

In [ ]:
setup_attempted = True
checks_verified = False
for index, statement in enumerate(setup_statements):
    execute(statement, f'setup_{index}')
print('Synthetic fixtures and read-only function created; not yet an MCP invocation.')

In [ ]:
checks_verified = False
rows = execute(f'SELECT CASE_ID, STATUS, PROCEEDS, ALLOCATED_TOTAL, GAP FROM {namespace}.WATERFALL_RECONCILIATION ORDER BY CASE_ID', 'direct_reconciliation_check')
expected = {
    'BALANCED': ('checks_passed', Decimal('0')),
    'SHORTFALL': ('blocked_reconciliation', Decimal('63000000')),
    'OVERALLOCATED': ('blocked_reconciliation', Decimal('-1000000')),
    'MISSING': ('blocked_incomplete', None),
    'INCOMPLETE': ('blocked_incomplete', None),
}
assert len(rows) == 5
assert {row['CASE_ID'] for row in rows} == set(expected)
for row in rows:
    assert (row['STATUS'], row['GAP']) == expected[row['CASE_ID']]
    if row['GAP'] is None:
        assert row['ALLOCATED_TOTAL'] is None
function_results = {}
for case in (*expected, 'UNKNOWN'):
    # case values are from the fixed local allowlist above, not user input.
    result = execute(f"SELECT {namespace}.REVIEW_RECONCILIATION('{case}') AS RESULT", f'direct_function_{case}')[0]['RESULT']
    result = json.loads(result) if isinstance(result, str) else result
    assert result['synthetic'] is True and result['payment_authorized'] is False
    assert result['status'] == (expected[case][0] if case in expected else 'blocked_unknown_case')
    function_results[case] = result
checks_verified = True

In [ ]:
mcp_name = namespace + '.FINANCIAL_VERIFICATION_MCP'
mcp_specification = f'''tools:
  - name: review_distribution_reconciliation
    title: Review synthetic distribution reconciliation
    type: GENERIC
    identifier: {namespace}.REVIEW_RECONCILIATION
    description: Review one synthetic distribution case for completeness and conservation. Returns evidence and blocking status. Does not validate recipient entitlement or authorize payments.
    config:
      type: function
      warehouse: {warehouse}
      query_timeout: 30
      input_schema:
        type: object
        properties:
          P_CASE_ID:
            type: string
            description: Synthetic case identifier; unknown cases are blocked.
        required: [P_CASE_ID]
        additionalProperties: false
'''
create_mcp_sql = f'CREATE OR REPLACE MCP SERVER {mcp_name} FROM SPECIFICATION $$\n{mcp_specification}$$'
print(create_mcp_sql)
print('Review, then set APPROVE_CREATE_MCP = True in a separate cell before deployment.')

In [ ]:
import pandas as pd
if namespace != f'{identifier(DATABASE)}.{identifier(SCHEMA)}':
    raise RuntimeError('Configuration changed after review')
execute(create_mcp_sql, 'create_managed_mcp')
metadata = execute(f'DESCRIBE MCP SERVER {mcp_name}', 'inspect_managed_mcp')
display(pd.DataFrame([row.as_dict() for row in metadata]))
print('Managed MCP server created. No MCP client has been authenticated or invoked by this cell.')

In [ ]:
# Explicit local runtime export; download/persist these files before shutdown.
# This contains SQL execution evidence, NOT MCP protocol responses.
from pathlib import Path

def export_sql_evidence():
    output = Path('evidence') / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    output.mkdir(parents=True, exist_ok=False)
    (output / 'sql_execution_evidence.json').write_text(json.dumps(query_evidence, indent=2, default=str))
    (output / 'managed_mcp.yaml').write_text(mcp_specification)
    print(f'Saved to {output}; direct SQL evidence only. Review before sharing.')

print('Call export_sql_evidence() after execution to retain query IDs and results.')

In [ ]:
import os
import time
import uuid
import re
import threading
import hashlib
from importlib.metadata import version
from urllib.parse import urlsplit
import requests

_acct_info = session.sql("SELECT CURRENT_ORGANIZATION_NAME() AS ORG, CURRENT_ACCOUNT_NAME() AS ACCT, CURRENT_USER() AS USR").collect()[0].as_dict()
MCP_ACCOUNT_HOST = f"{_acct_info['ORG']}-{_acct_info['ACCT']}".lower().replace('_', '-') + '.snowflakecomputing.com'

# PAT for MCP REST API authentication.
# Generate your own PAT (Snowsight > Settings > Authentication > Programmatic access tokens,
# or ALTER USER ... ADD PROGRAMMATIC ACCESS TOKEN), preferably role-restricted and short-lived.
# Supply it at runtime; never paste the secret into the notebook or commit it.
_PAT_PLACEHOLDER = '<Generate your own PAT>'
_pat_name = os.environ.get('FINVIEW_MCP_PAT_NAME', '<Your PAT name>')
_MCP_PAT_SECRET = os.environ.get('FINVIEW_MCP_PAT', _PAT_PLACEHOLDER)

APPROVE_MCP_DISCOVERY = False
APPROVE_TRULENS_RUN = False
APPROVE_JUDGE_METRICS = False
JUDGE_MODEL = ''
mcp_events = []
eval_attempted = False
metrics_attempted = False


def _get_auth_header():
    if not _MCP_PAT_SECRET or _MCP_PAT_SECRET == _PAT_PLACEHOLDER:
        raise ValueError('Generate your own PAT and supply it via FINVIEW_MCP_PAT before MCP calls')
    return 'Bearer ' + _MCP_PAT_SECRET


def parse_mcp_response(body, media_type, request_id):
    text = body.decode('utf-8')
    if media_type.split(';')[0].strip() == 'text/event-stream':
        messages = []
        for block in re.split(r'\r?\n\r?\n', text):
            data = '\n'.join(line[5:].lstrip(' ') for line in block.splitlines() if line.startswith('data:'))
            if data and data != '[DONE]':
                messages.append(json.loads(data))
    elif media_type.split(';')[0].strip() == 'application/json':
        messages = [json.loads(text)]
    else:
        raise ValueError('Unexpected MCP response content type')
    matches = [item for item in messages if isinstance(item, dict) and item.get('id') == request_id]
    if len(matches) != 1 or matches[0].get('jsonrpc') != '2.0':
        raise ValueError('Missing or ambiguous MCP response ID')
    message = matches[0]
    if 'error' in message or not isinstance(message.get('result'), dict):
        raise ValueError('MCP JSON-RPC error or missing result')
    return message


class SnowflakeMCPHTTP:
    def __init__(self, host, database, schema, max_calls=7):
        if not re.fullmatch(r'[a-z0-9][a-z0-9.-]*\.snowflakecomputing\.com', host):
            raise ValueError('Set a verified HTTPS Snowflake hostname without underscores')
        self.endpoint = f'https://{host}/api/v2/databases/{identifier(database)}/schemas/{identifier(schema)}/mcp-servers/FINANCIAL_VERIFICATION_MCP'
        self.max_calls = max_calls
        self.calls = 0
        self.lock = threading.Lock()
        self.discovered = False

    def request(self, method, params=None):
        with self.lock:
            if self.calls >= self.max_calls:
                raise RuntimeError('MCP request budget exhausted; no automatic reruns')
            self.calls += 1
        request_id = str(uuid.uuid4())
        payload = {'jsonrpc': '2.0', 'id': request_id, 'method': method}
        if params is not None:
            payload['params'] = params
        event = {'request': payload, 'endpoint': self.endpoint,
                 'utc': datetime.now(timezone.utc).isoformat(), 'execution_kind': 'live_mcp_http'}
        started = time.monotonic()
        try:
            auth_header = _get_auth_header()
            with requests.Session() as http:
                http.trust_env = False
                with http.post(self.endpoint, json=payload,
                               headers={'Authorization': auth_header,
                                        'X-Snowflake-Authorization-Token-Type': 'PROGRAMMATIC_ACCESS_TOKEN',
                                        'Accept': 'application/json, text/event-stream'},
                               timeout=(10, 60), allow_redirects=False, stream=True) as response:
                    event['http_status'] = response.status_code
                    if response.status_code != 200:
                        raise RuntimeError(f'MCP HTTP request rejected (status {response.status_code})')
                    body = bytearray()
                    for chunk in response.iter_content(chunk_size=8192):
                        body.extend(chunk)
                        if len(body) > 1024 * 1024 or time.monotonic() - started > 90:
                            raise RuntimeError('MCP response exceeded size/time budget')
                    message = parse_mcp_response(bytes(body), response.headers.get('Content-Type', ''), request_id)
            event['response'] = message
            if message['result'].get('isError'):
                raise RuntimeError('MCP tool returned an error')
            event['status'] = 'ok'
            return message['result']
        except Exception as exc:
            event.update(status='error', error_type=type(exc).__name__)
            raise RuntimeError('MCP failed; inspect sanitized mcp_events and client configuration') from None
        finally:
            event['latency_ms'] = (time.monotonic() - started) * 1000
            with self.lock:
                mcp_events.append(event)

    def discover(self):
        self.discovered = False
        result = self.request('tools/list')
        tools = result.get('tools', [])
        if result.get('nextCursor') or [tool.get('name') for tool in tools] != ['review_distribution_reconciliation']:
            raise ValueError('Review unexpected/paginated tool inventory; no tool calls enabled')
        schema = tools[0].get('inputSchema', {})
        if (schema.get('type') != 'object' or set(schema.get('properties', {})) != {'P_CASE_ID'}
                or schema['properties']['P_CASE_ID'].get('type') != 'string'
                or schema.get('required') != ['P_CASE_ID']):
            raise ValueError('Unexpected tool input schema')
        self.discovered = True
        return result

    def review(self, case_id):
        if not self.discovered:
            raise RuntimeError('Verify live tool discovery first')
        if case_id not in {'BALANCED', 'SHORTFALL', 'OVERALLOCATED', 'MISSING', 'INCOMPLETE', 'UNKNOWN'}:
            raise ValueError('Only synthetic cases are permitted')
        result = self.request('tools/call', {'name': 'review_distribution_reconciliation', 'arguments': {'P_CASE_ID': case_id}})
        return json.dumps(result, sort_keys=True)

print(f'Client defined; MCP_ACCOUNT_HOST = {MCP_ACCOUNT_HOST}, PAT = {_pat_name}')

In [ ]:
if 'mcp_client' in globals():
    del mcp_client
mcp_client = SnowflakeMCPHTTP(MCP_ACCOUNT_HOST, DATABASE, SCHEMA, max_calls=7)
inventory = mcp_client.discover()
display(inventory)
print('Live tool discovery complete. Six tool calls remain in the client budget.')

In [ ]:
%pip install "sqlalchemy>=2.0,<2.1" trulens-core trulens-connectors-snowflake trulens-providers-cortex snowflake-sqlalchemy fastembed -q

In [ ]:
# Imports and instrumented class definitions only; registration is in the next gated cell.
import pandas as pd
os.environ['TRULENS_OTEL_TRACING'] = '1'
from trulens.apps.app import TruApp
from trulens.connectors.snowflake import SnowflakeConnector
from trulens.core import TruSession
from trulens.core.run import RunConfig
from trulens.core.otel.instrument import instrument
from trulens.otel.semconv.trace import SpanAttributes

package_versions = {name: version(name) for name in
                    ('trulens-core', 'trulens-connectors-snowflake', 'trulens-providers-cortex', 'requests')}
print(package_versions)

class FinancialMCPApplication:
    @instrument(span_type=SpanAttributes.SpanType.RETRIEVAL,
                attributes={SpanAttributes.RETRIEVAL.QUERY_TEXT: 'query',
                            SpanAttributes.RETRIEVAL.RETRIEVED_CONTEXTS: 'return'})
    def retrieve(self, query: str, gold: str = None) -> list:
        if query not in {'BALANCED', 'SHORTFALL', 'OVERALLOCATED', 'MISSING', 'INCOMPLETE', 'UNKNOWN'}:
            raise ValueError('Unknown evaluation input')
        return [mcp_client.review(query)]

    @instrument(attributes={SpanAttributes.RECORD_ROOT.INPUT: 'query',
                            SpanAttributes.RECORD_ROOT.OUTPUT: 'return'})
    def respond(self, query: str, gold: str = None) -> str:
        return self.retrieve(query)[0]

# Independently authored ground truth; never pass it to the MCP tool.
evaluation_df = pd.DataFrame([
    {'query': 'BALANCED', 'gold': 'Synthetic USD proceeds 300000000, allocated_total 300000000, gap 0, status checks_passed, payment_authorized false.'},
    {'query': 'SHORTFALL', 'gold': 'Synthetic USD proceeds 300000000, allocated_total 237000000, gap 63000000, status blocked_reconciliation, payment_authorized false.'},
    {'query': 'OVERALLOCATED', 'gold': 'Synthetic USD proceeds 300000000, allocated_total 301000000, gap -1000000, status blocked_reconciliation, payment_authorized false.'},
    {'query': 'MISSING', 'gold': 'Synthetic missing allocations, actual_count 0, expected_count 2, allocated_total null, gap null, status blocked_incomplete, payment_authorized false.'},
    {'query': 'INCOMPLETE', 'gold': 'Synthetic actual_count 2, null_amount_count 1, allocated_total null, gap null, status blocked_incomplete, payment_authorized false.'},
    {'query': 'UNKNOWN', 'gold': 'Status blocked_unknown_case, synthetic true, payment_authorized false.'},
])
dataset_hash = hashlib.sha256(evaluation_df.to_json(orient='records').encode()).hexdigest()
display(evaluation_df)
print('Six live MCP calls; correctness and groundedness judging are separately gated. No responder LLM is used.')

In [ ]:
import os
# Set MCP_OAUTH_TOKEN in your environment before starting the notebook; never paste the value here.
MCP_OAUTH_TOKEN = os.environ.get('MCP_OAUTH_TOKEN', '')
print('MCP_OAUTH_TOKEN set:', bool(MCP_OAUTH_TOKEN))

In [ ]:
session.sql(f'USE DATABASE {identifier(DATABASE)}').collect()
session.sql(f'USE SCHEMA {identifier(SCHEMA)}').collect()
connector = SnowflakeConnector(snowpark_session=session, use_account_event_table=True)
tru_session = TruSession(connector=connector)
application = FinancialMCPApplication()
APP_NAME = 'FINANCIAL_MCP_CLIENT_EVAL'
APP_VERSION = 'mcp_http_v1'
RUN_NAME = 'MCP_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:8]
tru_app = TruApp(application, app_name=APP_NAME, app_version=APP_VERSION,
                 connector=connector, main_method=application.respond)
run = tru_app.add_run(run_config=RunConfig(
    run_name=RUN_NAME, description='Six synthetic cases through live Snowflake-managed MCP; dataset sha256 ' + dataset_hash,
    label='managed-mcp-synthetic-v1', source_type='DATAFRAME', dataset_name='synthetic_reconciliation_v1',
    dataset_spec={'RECORD_ROOT.INPUT': 'query', 'RETRIEVAL.QUERY_TEXT': 'query',
                  'RECORD_ROOT.GROUND_TRUTH_OUTPUT': 'gold'}, llm_judge_name=JUDGE_MODEL))
print({'app': APP_NAME, 'version': APP_VERSION, 'run': RUN_NAME, 'dataset_hash': dataset_hash})
try:
    run.start(input_df=evaluation_df)
finally:
    from opentelemetry import trace as otel_trace
    provider = otel_trace.get_tracer_provider()
    if hasattr(provider, 'force_flush'):
        provider.force_flush()
    display(pd.DataFrame([{key: event.get(key) for key in ('utc', 'status', 'http_status', 'latency_ms', 'error_type')}
                          for event in mcp_events]))

import time as _time
for _i in range(24):
    _time.sleep(5)
    _status = run.get_status()
    _status_name = getattr(_status, 'name', str(_status).split('.')[-1])
    if (_i + 1) % 6 == 0:
        print(f'Poll {_i+1}: {_status_name}')
    if _status_name in {'INVOCATION_COMPLETED', 'INVOCATION_PARTIALLY_COMPLETED'}:
        print(f'Poll {_i+1}: {_status_name}')
        break
print(f'Final status: {run.get_status()}')
metrics_attempted = False

In [ ]:
# Read status; rerun this cell to refresh. Never rerun run.start() to poll.
print(run.get_status())
display(run.describe())

def submit_judge_metrics():
    global APPROVE_JUDGE_METRICS, metrics_attempted
    if not APPROVE_JUDGE_METRICS:
        raise PermissionError('Approve Cortex judge costs before computing metrics')
    APPROVE_JUDGE_METRICS = False
    checked_context()
    status = run.get_status()
    status_name = getattr(status, 'name', str(status).split('.')[-1])
    if status_name in {'INVOCATION_COMPLETED', 'INVOCATION_PARTIALLY_COMPLETED'}:
        pass
    elif status_name == 'CREATED':
        print(f'Warning: run status is {status_name} (OTEL traces may not have been ingested). Attempting metrics anyway.')
    else:
        raise RuntimeError(f'Invocation not ready: {status_name}')
    if metrics_attempted:
        raise RuntimeError('Metrics already attempted; inspect status before taking further action')
    metrics_attempted = True
    return run.compute_metrics(metrics=['correctness', 'groundedness'])

print('Review traces first. Then set APPROVE_JUDGE_METRICS=True and call submit_judge_metrics().')

In [ ]:
# Explicit export only; protocol responses contain synthetic tool data, not auth headers.
from pathlib import Path

def export_mcp_evaluation_evidence():
    output = Path('evidence') / ('mcp_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
    output.mkdir(parents=True, exist_ok=False)
    (output / 'mcp_protocol.json').write_text(json.dumps(mcp_events, indent=2))
    metadata = {'app': APP_NAME, 'version': APP_VERSION, 'run': RUN_NAME,
                'dataset_sha256': dataset_hash, 'packages': package_versions,
                'judge': JUDGE_MODEL, 'status': str(run.get_status()), 'endpoint': mcp_client.endpoint}
    (output / 'evaluation_metadata.json').write_text(json.dumps(metadata, indent=2))
    (output / 'run_description.json').write_text(json.dumps(run.describe(), indent=2, default=str))
    print(f'Saved {output}. Download before runtime termination; review before publication.')

print('Call export_mcp_evaluation_evidence() to retain actual protocol and evaluation metadata.')

In [ ]:
# Standalone synthetic retrieval/decision track. No network or Snowflake calls.
import copy
import hashlib
import json
import math
import os
import time
from datetime import date, datetime, timezone
from decimal import Decimal, InvalidOperation
from importlib.metadata import version

APPROVE_COLBERT_DOWNLOAD = False
APPROVE_SNOWFLAKE_DECISION = False
APPROVED_DECISION_PAYLOAD_SHA256 = {}  # Populate after reviewing decision_requests below.
COLBERT_MODEL = 'colbert-ir/colbertv2.0'
DECISION_MODEL = 'snowflake-decision'  # Public preview; runs in-account through AI_COMPLETE.
RRF_CONSTANT = 60

FINANCIAL_CASES = {
    'PE_ADDBACK': {
        'domain': 'PE', 'entity': 'SYNTHETIC_PORTCO', 'as_of': '2026-09-30',
        'question': 'Does the dated evidence support including the restructuring add-back in FY2025 covenant EBITDA?',
        'facts': {'reported_ebitda': '20000000', 'addback': '3000000',
                  'documented_eligible_cost': '2000000', 'currency': 'USD',
                  'period': 'FY2025', 'definition': 'covenant_ebitda'},
        'choice_criteria': {
            'supported': 'All of the proposed adjustment is supported by applicable dated evidence.',
            'review_required': 'A discrepancy or contractual interpretation requires human review.',
            'insufficient_evidence': 'The supplied evidence cannot support a conclusion.'},
        'keyword_ranking': ['pe_schedule', 'pe_terms', 'pe_background'],
        'vector_ranking': ['pe_terms', 'pe_background', 'pe_schedule'],
    },
    'CM_SETTLEMENT': {
        'domain': 'CAPITAL_MARKETS', 'entity': 'SYNTHETIC_BOND', 'as_of': '2026-09-30',
        'question': 'Which explanation best fits the settlement break for the synthetic bond: timing, price, quantity or insufficient evidence?',
        'facts': {'expected_quantity': '1000', 'actual_quantity': '1000',
                  'expected_price': '99.5', 'actual_price': '99.5',
                  'expected_date': '2026-09-29', 'actual_date': '2026-09-30',
                  'price_basis': 'clean_per_100', 'currency': 'USD'},
        'choice_criteria': {
            'timing': 'Settlement dates differ while quantity and price agree.',
            'price': 'Prices differ on a common basis.',
            'quantity': 'Quantities differ.',
            'insufficient_evidence': 'Evidence is missing or does not isolate a cause.'},
        'keyword_ranking': ['cm_ledger', 'cm_terms', 'cm_background'],
        'vector_ranking': ['cm_terms', 'cm_background', 'cm_ledger'],
    },
}

FINANCIAL_PASSAGES = [
    {'id': 'pe_terms', 'domain': 'PE', 'entity': 'SYNTHETIC_PORTCO', 'available_on': '2026-03-01',
     'text': 'Synthetic FY2025 covenant terms permit only documented eligible restructuring costs. Reported EBITDA and covenant EBITDA are different definitions.'},
    {'id': 'pe_schedule', 'domain': 'PE', 'entity': 'SYNTHETIC_PORTCO', 'available_on': '2026-03-02',
     'text': 'Synthetic FY2025 schedule: reported EBITDA USD 20m; proposed restructuring add-back USD 3m; documented eligible restructuring costs USD 2m.'},
    {'id': 'pe_background', 'domain': 'PE', 'entity': 'SYNTHETIC_PORTCO', 'available_on': '2026-02-01',
     'text': 'Synthetic portfolio overview discusses growth plans but contains no covenant-specific add-back schedule.'},
    {'id': 'cm_terms', 'domain': 'CAPITAL_MARKETS', 'entity': 'SYNTHETIC_BOND', 'available_on': '2026-09-01',
     'text': 'Synthetic bond confirmation requires settlement on 2026-09-29. Price is clean per 100; USD. Do not infer net cash from clean price without accrued interest and fees.'},
    {'id': 'cm_ledger', 'domain': 'CAPITAL_MARKETS', 'entity': 'SYNTHETIC_BOND', 'available_on': '2026-09-30',
     'text': 'Synthetic expected and actual quantity: 1000 and 1000; clean price: 99.5 and 99.5; expected date 2026-09-29, actual date 2026-09-30.'},
    {'id': 'cm_background', 'domain': 'CAPITAL_MARKETS', 'entity': 'SYNTHETIC_BOND', 'available_on': '2026-09-01',
     'text': 'Synthetic instrument overview mentions settlement conventions without identifying a break cause.'},
]


def stable_sha256(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(',', ':'),
                                     allow_nan=False).encode()).hexdigest()


def fuse_rankings(rankings, constant=60):
    if type(constant) is not int or constant <= 0:
        raise ValueError('RRF constant must be a positive integer')
    totals, ranks = {}, {}
    for name, ranking in rankings.items():
        if len(ranking) != len(set(ranking)):
            raise ValueError('Duplicate passage within a ranking')
        for rank, passage_id in enumerate(ranking, 1):
            totals[passage_id] = totals.get(passage_id, 0.0) + 1 / (constant + rank)
            ranks.setdefault(passage_id, {})[name] = rank
    return [{'id': passage_id, 'rrf_score': totals[passage_id], 'ranks': ranks[passage_id]}
            for passage_id in sorted(totals, key=lambda key: (-totals[key], key))]


def financial_candidates(case, passages):
    if len({item['id'] for item in passages}) != len(passages):
        raise ValueError('Duplicate passage ID')
    eligible = {item['id']: item for item in passages
                if item['domain'] == case['domain'] and item['entity'] == case['entity']
                and date.fromisoformat(item['available_on']) <= date.fromisoformat(case['as_of'])}
    rankings = {name: [key for key in case[name + '_ranking'] if key in eligible]
                for name in ('keyword', 'vector')}
    return [dict(eligible[row['id']], **{key: value for key, value in row.items() if key != 'id'})
            for row in fuse_rankings(rankings, RRF_CONSTANT)]


ranked_financial_evidence = {key: financial_candidates(case, FINANCIAL_PASSAGES)
                             for key, case in FINANCIAL_CASES.items()}
selected_financial_evidence = {key: rows[:2] for key, rows in ranked_financial_evidence.items()}
retrieval_execution = {'kind': 'fixture_rankings_local_rrf', 'constant': RRF_CONSTANT,
                       'fixture_sha256': stable_sha256([FINANCIAL_CASES, FINANCIAL_PASSAGES])}
financial_decision_events = []
print(json.dumps({'retrieval': retrieval_execution, 'selected': selected_financial_evidence}, indent=2))

In [ ]:
def colbert_maxsim(query_vectors, passage_vectors):
    import numpy as np
    query = np.asarray(query_vectors, dtype=float)
    passage = np.asarray(passage_vectors, dtype=float)
    if (query.ndim != 2 or passage.ndim != 2 or min(query.shape + passage.shape) == 0
            or query.shape[1] != passage.shape[1]
            or not np.isfinite(query).all() or not np.isfinite(passage).all()):
        raise ValueError('Expected nonempty finite token matrices with matching dimensions')
    score = float((query @ passage.T).max(axis=1).sum())
    if not math.isfinite(score):
        raise ValueError('Nonfinite MaxSim score')
    return score


def run_colbert_reranking():
    global APPROVE_COLBERT_DOWNLOAD, selected_financial_evidence, retrieval_execution
    if not APPROVE_COLBERT_DOWNLOAD:
        raise PermissionError('Review and approve the model download before local inference')
    APPROVE_COLBERT_DOWNLOAD = False
    from fastembed import LateInteractionTextEmbedding
    started = time.monotonic()
    event = {'execution_kind': 'local_colbert_inference', 'model': COLBERT_MODEL,
             'utc': datetime.now(timezone.utc).isoformat()}
    try:
        model = LateInteractionTextEmbedding(COLBERT_MODEL)
        passage_by_id = {row['id']: row for rows in ranked_financial_evidence.values() for row in rows}
        ids = sorted(passage_by_id)
        vectors = list(model.embed([passage_by_id[key]['text'] for key in ids]))
        if len(vectors) != len(ids):
            raise ValueError('Unexpected document embedding count')
        vector_by_id = dict(zip(ids, vectors))
        reranked = {}
        for key, case in FINANCIAL_CASES.items():
            query_vectors = list(model.query_embed([case['question']]))
            if len(query_vectors) != 1:
                raise ValueError('Unexpected query embedding count')
            rows = [dict(row, colbert_score=colbert_maxsim(query_vectors[0], vector_by_id[row['id']]))
                    for row in ranked_financial_evidence[key]]
            reranked[key] = sorted(rows, key=lambda row: (-row['colbert_score'], row['id']))
        event.update(status='ok', packages={'fastembed': version('fastembed'), 'numpy': version('numpy')},
                     rankings=reranked, artifact_revision='not_pinned')
        selected_financial_evidence = {key: rows[:2] for key, rows in reranked.items()}
        retrieval_execution = {'kind': 'fixture_rrf_then_live_local_colbert', 'model': COLBERT_MODEL,
                               'rankings_sha256': stable_sha256(reranked)}
        return reranked
    except Exception as exc:
        event.update(status='error', error_type=type(exc).__name__)
        raise RuntimeError('ColBERT failed; no new ranking selected. Inspect runtime configuration.') from None
    finally:
        event['latency_ms'] = (time.monotonic() - started) * 1000
        financial_decision_events.append(event)


print('Optional ColBERT function defined; no import, download or inference attempted.')

In [ ]:
APPROVE_JUDGE_METRICS=True

In [ ]:
submit_judge_metrics()

In [ ]:
export_mcp_evaluation_evidence()

In [ ]:
APPROVE_COLBERT_DOWNLOAD = True
colbert_rankings = run_colbert_reranking()

In [ ]:
# Judge track: AI_COMPLETE with the snowflake-decision model. No external service or API key.
# Evaluation-only labels: deliberately absent from prepare_decision_requests().
FINANCIAL_GOLD = {'PE_ADDBACK': 'review_required', 'CM_SETTLEMENT': 'timing'}
if 'decision_results' not in globals():
    decision_results = {}


def deterministic_financial_check(key, facts):
    """Rule-based expectation from the facts alone, recorded beside the model's answer."""
    if key == 'PE_ADDBACK':
        addback = Decimal(facts['addback'])
        documented = Decimal(facts['documented_eligible_cost'])
        return {'check': 'addback_vs_documented_cost', 'addback': str(addback),
                'documented_eligible_cost': str(documented), 'unsupported': str(addback - documented),
                'expected': 'supported' if addback <= documented else 'review_required'}
    if key == 'CM_SETTLEMENT':
        differs = {'quantity': Decimal(facts['expected_quantity']) != Decimal(facts['actual_quantity']),
                   'price': Decimal(facts['expected_price']) != Decimal(facts['actual_price']),
                   'timing': facts['expected_date'] != facts['actual_date']}
        causes = [name for name, flag in differs.items() if flag]
        return {'check': 'settlement_break_fields', 'differs': differs,
                'expected': causes[0] if len(causes) == 1 else 'insufficient_evidence'}
    raise KeyError(key)


def prepare_decision_requests():
    requests = {}
    for key, case in FINANCIAL_CASES.items():
        evidence = [{'id': row['id'], 'available_on': row['available_on'], 'text': row['text']}
                    for row in selected_financial_evidence[key]]
        requests[key] = {
            'state': json.dumps({'question': case['question'], 'as_of': case['as_of'],
                                 'facts': case['facts'], 'evidence': evidence}, sort_keys=True),
            'questions': {'assessment': {'type': 'choice', 'instructions': case['question'],
                                         'criteria': case['choice_criteria']}},
        }
    return requests


def validate_decision_response(response, payload):
    answer = response['answers']['assessment']
    options = set(payload['questions']['assessment']['criteria'])
    probabilities = {key: float(value) for key, value in answer['probabilities'].items()}
    if answer.get('type') != 'choice' or answer['choice'] not in options or set(probabilities) != options:
        raise ValueError('Decision response does not match the requested choice question')
    if not all(0.0 <= value <= 1.0 for value in probabilities.values()) or abs(sum(probabilities.values()) - 1) > 1e-3:
        raise ValueError('Decision probabilities are not a valid distribution')
    return {'answers': {'assessment': dict(answer, probabilities=probabilities)}}


decision_requests = prepare_decision_requests()
decision_payload_hashes = {key: stable_sha256(payload) for key, payload in decision_requests.items()}


def run_snowflake_decisions():
    global APPROVE_SNOWFLAKE_DECISION
    if not APPROVE_SNOWFLAKE_DECISION:
        raise PermissionError('Review decision_requests, then approve the billable AI_COMPLETE calls')
    APPROVE_SNOWFLAKE_DECISION = False
    current = prepare_decision_requests()
    for key, payload in current.items():
        payload_sha256 = stable_sha256(payload)
        if APPROVED_DECISION_PAYLOAD_SHA256.get(key) != payload_sha256:
            raise PermissionError(f'{key}: payload changed since approval; review and re-approve')
        event = {'execution_kind': 'snowflake_ai_complete', 'model': DECISION_MODEL, 'case_id': key,
                 'payload_sha256': payload_sha256, 'utc': datetime.now(timezone.utc).isoformat()}
        started = time.monotonic()
        try:
            with session.query_history() as history:
                row = session.sql(f"SELECT AI_COMPLETE(model => '{DECISION_MODEL}', prompt => ?) AS J",
                                  params=[json.dumps(payload, sort_keys=True)]).collect()[0]
            raw = row[0]
            response = validate_decision_response(json.loads(raw) if isinstance(raw, str) else raw, payload)
            decision_results[key] = {'payload_sha256': payload_sha256, 'response': response}
            event.update(status='ok', query_ids=[query.query_id for query in history.queries])
        except Exception as exc:
            event.update(status='error', error_type=type(exc).__name__, error=str(exc)[:300])
            raise
        finally:
            event['latency_ms'] = (time.monotonic() - started) * 1000
            financial_decision_events.append(event)


def summarize_financial_decisions():
    if set(decision_results) != set(FINANCIAL_GOLD):
        return {'status': 'NOT_EVALUATED', 'reason': 'Requires both validated snowflake-decision results',
                'completed_cases': sorted(decision_results), 'calibration_established': False}
    current_payloads = prepare_decision_requests()
    rows = []
    for key, gold in FINANCIAL_GOLD.items():
        result = decision_results[key]
        if result['payload_sha256'] != stable_sha256(current_payloads[key]):
            raise ValueError('Stale decision response: current evidence, facts or questions differ')
        answer = validate_decision_response(result['response'], current_payloads[key])['answers']['assessment']
        probabilities = answer['probabilities']
        rows.append({'case_id': key, 'gold': gold, 'choice': answer['choice'],
                     'correct': answer['choice'] == gold,
                     'deterministic_expected': deterministic_financial_check(key, FINANCIAL_CASES[key]['facts'])['expected'],
                     'brier_sum_classes': sum((value - int(option == gold)) ** 2
                                              for option, value in probabilities.items()),
                     'probabilities': probabilities, 'provider_confidence': answer['confidence'],
                     'model': DECISION_MODEL, 'action_authorized': False})
    return {'status': 'SYNTHETIC_SMOKE_TEST_ONLY', 'rows': rows, 'cases': len(rows),
            'accuracy': sum(row['correct'] for row in rows) / len(rows),
            'brier_mean_sum_classes': sum(row['brier_sum_classes'] for row in rows) / len(rows),
            'calibration_established': False}


def export_financial_decision_evidence():
    from pathlib import Path
    record = {'retrieval': retrieval_execution, 'selected_evidence': selected_financial_evidence,
              'current_payloads': prepare_decision_requests(), 'provider_results': decision_results,
              'events': financial_decision_events,
              'deterministic_checks': {key: deterministic_financial_check(key, case['facts'])
                                       for key, case in FINANCIAL_CASES.items()},
              'evaluation': summarize_financial_decisions(), 'action_authorized': False}
    output = Path('evidence') / ('financial_decisions_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
    output.mkdir(parents=True, exist_ok=False)
    (output / 'financial_decisions.json').write_text(json.dumps(record, indent=2, allow_nan=False))
    return str(output)


print(json.dumps({'decision_payload_sha256': decision_payload_hashes,
                  'deterministic_checks': {key: deterministic_financial_check(key, case['facts'])
                                           for key, case in FINANCIAL_CASES.items()}}, indent=2))
print(json.dumps(summarize_financial_decisions(), indent=2))

In [ ]:
# Billable: two AI_COMPLETE calls with the snowflake-decision model in your account.
# Review decision_requests from the previous cell before approving.
APPROVED_DECISION_PAYLOAD_SHA256 = dict(decision_payload_hashes)
APPROVE_SNOWFLAKE_DECISION = True

run_snowflake_decisions()
print(json.dumps(summarize_financial_decisions(), indent=2))

In [ ]:
from pathlib import Path
import hashlib
import json
import pandas as pd
from IPython.display import display, Markdown, Code

SKILL_BUNDLE_DIR = Path('pe-agentic-system/skills/generated').resolve()
skill_manifest = json.loads((SKILL_BUNDLE_DIR / 'manifest.json').read_text())

def verified_skill_artifact(relative):
    path = (SKILL_BUNDLE_DIR / relative).resolve()
    if not path.is_relative_to(SKILL_BUNDLE_DIR):
        raise ValueError('File is outside the bundle.')
    if hashlib.sha256(path.read_bytes()).hexdigest() != skill_manifest['files'][relative]:
        raise RuntimeError(f'Changed bundle file: {relative}')
    return path

skill_inventory = pd.DataFrame(skill_manifest['coco_skills'])
for skill in skill_manifest['coco_skills']:
    verified_skill_artifact(skill['path'])
display(skill_inventory[['name', 'status', 'description']])
print('11 actual instruction packages. Loading or displaying them is not CoCo invocation.')

In [ ]:
import importlib
import sys

support_root = (SKILL_BUNDLE_DIR / 'supporting_code').resolve()
for relative in skill_manifest['files']:
    if relative.startswith('supporting_code/'):
        verified_skill_artifact(relative)
for module_name in ('assurance', 'assurance.contracts', 'examples', 'examples.assurance_boundary_demo', 'examples.coco_skill_demo'):
    loaded = sys.modules.get(module_name)
    if loaded is not None and (not getattr(loaded, '__file__', None) or not Path(loaded.__file__).resolve().is_relative_to(support_root)):
        raise RuntimeError(f'{module_name} was loaded from another location. Restart the kernel before this demo.')
sys.path.insert(0, str(support_root))
demo_module = importlib.import_module('examples.coco_skill_demo')
supporting_demo_result = demo_module.run_demo()
supporting_demo_rows = pd.DataFrame([
    {'Case': result['case'], 'Status': result['status'], 'Findings': len(result['findings']),
     'Input SHA-256': result['input_hash'], 'Execution': result['execution_kind']}
    for result in supporting_demo_result['cases']
])
display(Markdown('### Valuation-input checks: actual supporting-code output'))
display(supporting_demo_rows)
print(supporting_demo_result['limitations'])

In [ ]:
from pathlib import Path
import hashlib
import io
import json
import re
from datetime import datetime, timezone
import pandas as pd
from IPython.display import display, Markdown
from snowflake.snowpark.context import get_active_session
from snowflake.connector.util_text import split_statements

BUNDLE_DIR = Path('pe-agentic-system/skills/generated')
EXPECTED_ACCOUNT = '<your account>'
DATABASE = '<your db>'
SCHEMA = '<your schema>'
ROLE = '<your role>'
WAREHOUSE = '<your warehouse>'
SETUP_MODE = 'reuse_finview'  # 'fresh' only for a new schema; never replaces existing data.
APPROVE_SETUP = False
APPROVE_SKILL_UPLOAD_AND_AGENT = False
APPROVE_ONE_BILLABLE_EVALUATION = False
APPROVE_CANCEL = False

session = get_active_session()
manifest = json.loads((BUNDLE_DIR / 'manifest.json').read_text())
if manifest['account_name'] != EXPECTED_ACCOUNT:
    raise RuntimeError('Regenerate the bundle for the intended account.')
if not re.fullmatch(r'[A-Z_][A-Z0-9_]*\.[A-Z_][A-Z0-9_]*', manifest['namespace']):
    raise ValueError('Expected simple standard database.schema identifiers.')
if (manifest['namespace'], manifest['warehouse'], manifest.get('role')) != (DATABASE + '.' + SCHEMA, WAREHOUSE, ROLE):
    raise RuntimeError('Bundle target differs from notebook configuration; regenerate it before proceeding.')
namespace = DATABASE + '.' + SCHEMA
database_name, schema_name = DATABASE, SCHEMA
warehouse_name = WAREHOUSE
if not re.fullmatch(r'[A-Z_][A-Z0-9_]*', warehouse_name):
    raise ValueError('Invalid warehouse identifier.')
for relative_name, expected_hash in manifest['files'].items():
    artifact = (BUNDLE_DIR / relative_name).resolve()
    if not artifact.is_relative_to(BUNDLE_DIR.resolve()):
        raise ValueError('Artifact is outside the bundle.')
    if hashlib.sha256(artifact.read_bytes()).hexdigest() != expected_hash:
        raise RuntimeError(f'Bundle hash mismatch: {relative_name}')
evidence = globals().get('evidence', [])
setup_attempted = globals().get('setup_attempted', False)

def checked_context(require_role=True):
    row = session.sql('SELECT CURRENT_ACCOUNT_NAME() AS ACCOUNT_NAME, CURRENT_USER() AS USER_NAME, CURRENT_ROLE() AS ROLE_NAME, CURRENT_WAREHOUSE() AS WAREHOUSE_NAME').collect()[0].as_dict()
    if row['ACCOUNT_NAME'] != EXPECTED_ACCOUNT:
        raise RuntimeError(f"Wrong account: {row['ACCOUNT_NAME']}. Switch notebook account before continuing.")
    if require_role and (row['ROLE_NAME'] != ROLE or row['WAREHOUSE_NAME'] != WAREHOUSE):
        raise RuntimeError('Active role/warehouse changed; restore the configured demo context.')
    return row

def execute_recorded(statement, label):
    checked_context()
    with session.query_history() as history:
        rows = session.sql(statement).collect()
    evidence.append({'label': label, 'utc': datetime.now(timezone.utc).isoformat(),
                     'query_ids': [record.query_id for record in history.queries],
                     'rows': [row.as_dict() for row in rows]})
    return pd.DataFrame([row.as_dict() for row in rows])

def select_namespace():
    execute_recorded(f'USE DATABASE {database_name}', 'database context')
    execute_recorded(f'USE SCHEMA {namespace}', 'schema context')
    execute_recorded(f'USE WAREHOUSE {warehouse_name}', 'warehouse context')

def execute_file(filename):
    statements = list(split_statements(io.StringIO((BUNDLE_DIR / filename).read_text())))
    for position, (statement, _) in enumerate(statements, 1):
        if statement.strip():
            result = execute_recorded(statement, f'{filename}:{position}')
            display(result)

checked_context(require_role=False)
session.use_role(ROLE)
session.use_warehouse(WAREHOUSE)
context = checked_context()
display(pd.DataFrame([context]))
print('Verified bundle:', namespace, '| status:', manifest['status'])

In [ ]:
execute_file('00_preflight.sql')
for filename in ('01_setup.sql', '09_existing_finview_support.sql', 'agent.yaml', 'evaluation.yaml', 'skills/waterfall-reconciliation-review/SKILL.md'):
    display(Markdown(f'### {filename}'))
    print((BUNDLE_DIR / filename).read_text())

In [ ]:
if not APPROVE_SETUP:
    raise RuntimeError('Review setup SQL, select SETUP_MODE, then set APPROVE_SETUP=True in a separate cell.')
APPROVE_SETUP = False
checked_context()
if setup_attempted:
    raise RuntimeError('Setup already attempted. Inspect partial state before retrying; no destructive reset.')
if SETUP_MODE not in {'fresh', 'reuse_finview'}:
    raise ValueError('SETUP_MODE must be fresh or reuse_finview')
if SETUP_MODE == 'fresh':
    execute_recorded(f'CREATE DATABASE IF NOT EXISTS {database_name}', 'create database if absent')
database_info = execute_recorded(f"SHOW DATABASES LIKE '{database_name}'", 'database eligibility')
matching_databases = database_info[database_info['name'] == database_name]
if len(matching_databases) != 1 or matching_databases.iloc[0]['kind'] != 'STANDARD':
    raise RuntimeError('An approved STANDARD database is required')
if SETUP_MODE == 'fresh':
    existing_schemas = execute_recorded(f"SHOW SCHEMAS LIKE '{schema_name}' IN DATABASE {database_name}", 'schema collision')
    if not existing_schemas.empty and (existing_schemas['name'] == schema_name).any():
        raise RuntimeError('Schema already exists; use reuse_finview, not fresh setup.')
    setup_attempted = True
    execute_file('01_setup.sql')
else:
    select_namespace()
    # Validate existing finview fixtures before adding only the older demo support objects.
    statuses = execute_recorded(f'SELECT CASE_ID,STATUS,GAP,ALLOCATED_TOTAL FROM {namespace}.WATERFALL_RECONCILIATION ORDER BY CASE_ID', 'reuse fixture validation')
    from decimal import Decimal
    expected = {'BALANCED':('checks_passed',Decimal('0')), 'SHORTFALL':('blocked_reconciliation',Decimal('63000000')),
                'OVERALLOCATED':('blocked_reconciliation',Decimal('-1000000')), 'MISSING':('blocked_incomplete',None), 'INCOMPLETE':('blocked_incomplete',None)}
    if len(statuses) != 5 or set(statuses.CASE_ID) != set(expected):
        raise RuntimeError('Unexpected existing fixture set')
    for row in statuses.to_dict('records'):
        status, gap = expected[row['CASE_ID']]
        if row['STATUS'] != status or (not pd.isna(row['GAP']) if gap is None else Decimal(str(row['GAP'])) != gap):
            raise RuntimeError('Existing reconciliation values differ from expected fixtures')
        if gap is None and not pd.isna(row['ALLOCATED_TOTAL']):
            raise RuntimeError('Incomplete case must retain NULL allocation total')
    execute_recorded(f'DESCRIBE FUNCTION {namespace}.REVIEW_RECONCILIATION(VARCHAR)', 'existing function')
    existing_support = {
        'RECONCILIATION_FAILURE_COUNT': execute_recorded(f'SHOW DATA METRIC FUNCTIONS IN SCHEMA {namespace}', 'existing DMFs'),
        'YAML_FILE_FORMAT': execute_recorded(f'SHOW FILE FORMATS IN SCHEMA {namespace}', 'existing formats'),
        'DEMO_ASSETS': execute_recorded(f'SHOW STAGES IN SCHEMA {namespace}', 'existing stages'),
        'WATERFALL_EVAL_INPUT': execute_recorded(f'SHOW TABLES IN SCHEMA {namespace}', 'existing evaluation table'),
    }
    present = {name for name, frame in existing_support.items() if not frame.empty and name in set(frame['name'])}
    setup_attempted = True
    for statement, _ in split_statements(io.StringIO((BUNDLE_DIR / '09_existing_finview_support.sql').read_text())):
        if not statement.strip().strip(';'):
            continue
        target = next((name for name in existing_support if namespace + '.' + name in statement), None)
        if target is None:
            raise RuntimeError('Unexpected support statement')
        if target not in present:
            execute_recorded(statement, 'add_missing_support_' + target)
select_namespace()
print('Setup ready. Existing finview tables, function and MCP server were not replaced.')

In [ ]:
select_namespace()
execute_file('02_checks.sql')
reconciliation = execute_recorded(
    f'SELECT CASE_ID, PROCEEDS, EXPECTED_COUNT, ACTUAL_COUNT, NULL_AMOUNT_COUNT, ALLOCATED_TOTAL, GAP, STATUS FROM {namespace}.WATERFALL_RECONCILIATION ORDER BY CASE_ID LIMIT 20',
    'reconciliation screenshot data')
expected_status = {'BALANCED': 'checks_passed', 'SHORTFALL': 'blocked_reconciliation',
                   'OVERALLOCATED': 'blocked_reconciliation', 'MISSING': 'blocked_incomplete',
                   'INCOMPLETE': 'blocked_incomplete'}
if len(reconciliation) != 5 or dict(zip(reconciliation.CASE_ID, reconciliation.STATUS)) != expected_status:
    raise RuntimeError('Fixture results differ from expected statuses. Stop before agent creation.')
metrics = execute_recorded(
    f'SELECT {namespace}.RECONCILIATION_FAILURE_COUNT(SELECT PROCEEDS, ALLOCATED_TOTAL, TOLERANCE FROM {namespace}.WATERFALL_RECONCILIATION) AS VALUE',
    'direct DMF measurement')
if int(metrics.iloc[0]['VALUE']) != 4:
    raise RuntimeError('Expected four intentionally failed cases.')
display(reconciliation)
display(metrics)

In [ ]:
import matplotlib.pyplot as plt

figure, axes = plt.subplots(figsize=(13, 4.5))
axes.axis('off')
figure.suptitle('Synthetic distribution reconciliation | Recorded Snowflake results', fontsize=16, x=0.5)
rendered = reconciliation[['CASE_ID', 'ALLOCATED_TOTAL', 'GAP', 'ACTUAL_COUNT', 'STATUS']].copy()
for column in ('ALLOCATED_TOTAL', 'GAP'):
    rendered[column] = rendered[column].map(lambda value: 'Not established' if pd.isna(value) else f'${value:,.2f}')
table = axes.table(cellText=rendered.values, colLabels=['Case', 'Allocated', 'Signed gap', 'Rows', 'Check status'],
                   colWidths=[0.16, 0.21, 0.19, 0.07, 0.32], loc='center', cellLoc='left')
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 2)
for (row_index, column_index), cell in table.get_celld().items():
    cell.set_edgecolor('#d8e2ea')
    if row_index == 0:
        cell.set_facecolor('#eaf2f8')
        cell.set_text_props(weight='bold')
figure.text(0.05, 0.07, f"Direct DMF VALUE: {int(metrics.iloc[0]['VALUE'])} | Tolerance: $0.01 | No payment authorization", fontsize=10)
figure.text(0.05, 0.025, 'Notebook rendering of query results, not a Snowsight Evaluations screenshot or proof of skill invocation.', fontsize=9)
plt.show()

In [ ]:
if not APPROVE_SKILL_UPLOAD_AND_AGENT:
    raise RuntimeError('Review definitions and set APPROVE_SKILL_UPLOAD_AND_AGENT=True before running once.')
APPROVE_SKILL_UPLOAD_AND_AGENT = False
checked_context()
select_namespace()
skill_relative = 'skills/waterfall-reconciliation-review/SKILL.md'
skill_stage_file = f"@{namespace}.DEMO_ASSETS/skills/{manifest['skill_sha256']}/waterfall-reconciliation-review/SKILL.md"
config_stage_file = f'@{namespace}.DEMO_ASSETS/evaluation/v1/evaluation.yaml'
for relative_name, stage_file in ((skill_relative, skill_stage_file), ('evaluation.yaml', config_stage_file)):
    checked_context()
    content = (BUNDLE_DIR / relative_name).read_bytes()
    expected_hash = manifest['files'][relative_name]
    if hashlib.sha256(content).hexdigest() != expected_hash:
        raise RuntimeError('File changed since bundle verification.')
    upload = session.file.put_stream(io.BytesIO(content), stage_file, auto_compress=False, overwrite=False)
    with session.file.get_stream(stage_file, decompress=False) as downloaded:
        actual_hash = hashlib.sha256(downloaded.read()).hexdigest()
    if actual_hash != expected_hash:
        raise RuntimeError('Stage content differs. Stop; do not overwrite an existing skill silently.')
    evidence.append({'label': 'verified stage upload', 'file': stage_file, 'sha256': actual_hash,
                     'utc': datetime.now(timezone.utc).isoformat(), 'status': str(upload.status)})
execute_file('04_create_agent.sql')
print('Inspect the displayed spec and VERSION$1 before enabling evaluation.')

In [ ]:
if not APPROVE_ONE_BILLABLE_EVALUATION:
    raise RuntimeError('Billable evaluation disabled. Obtain approval and set the gate True to start once.')
APPROVE_ONE_BILLABLE_EVALUATION = False
checked_context()
select_namespace()
input_count = execute_recorded(f'SELECT COUNT(*) AS RECORD_COUNT FROM {namespace}.WATERFALL_EVAL_INPUT', 'evaluation input count')
if int(input_count.iloc[0]['RECORD_COUNT']) != 6:
    raise RuntimeError('Evaluation must have exactly six inputs.')
if globals().get('evaluation_start_attempted', False):
    raise RuntimeError('Start was already attempted in this kernel. Inspect status and logs; do not blindly retry.')
evaluation_start_attempted = True
execute_file('05_start_evaluation_BILLABLE.sql')

In [ ]:
select_namespace()
run_name = manifest['run_name']
if not re.fullmatch(r'[a-zA-Z0-9_]+', run_name):
    raise ValueError('Invalid run identifier.')
config_stage_file = f'@{namespace}.DEMO_ASSETS/evaluation/v1/evaluation.yaml'
display(execute_recorded(
    f"CALL EXECUTE_AI_EVALUATION('STATUS', OBJECT_CONSTRUCT('run_name', '{run_name}'), '{config_stage_file}')",
    'managed evaluation status'))
evaluation_results = execute_recorded(
    f"SELECT * FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_EVALUATION_DATA('{database_name}', '{schema_name}', 'WATERFALL_REVIEW_AGENT', 'CORTEX AGENT', '{run_name}')) LIMIT 100",
    'managed evaluation results')
if evaluation_results.empty:
    print('No result rows yet. Wait and rerun this read-only status/results cell; do not rerun START.')
else:
    visible_columns = [column for column in ('RECORD_ID', 'INPUT', 'OUTPUT', 'METRIC_NAME', 'EVAL_AGG_SCORE', 'ERROR') if column in evaluation_results.columns]
    display(evaluation_results[visible_columns])

In [ ]:
RECORD_ID = ''
if not re.fullmatch(r'[A-Za-z0-9_-]+', RECORD_ID):
    raise ValueError('Set RECORD_ID to one actual evaluation record identifier shown above.')
record_trace = execute_recorded(
    f"SELECT * FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_RECORD_TRACE('{database_name}', '{schema_name}', 'WATERFALL_REVIEW_AGENT', 'CORTEX AGENT', '{RECORD_ID}')) LIMIT 500",
    'selected evaluation record trace')
display(record_trace)
if len(record_trace) == 500:
    print('Trace reached the 500-row cap; it may be incomplete. Inspect remaining trace data before drawing conclusions.')

In [ ]:
OUTPUT_DIR = Path('evidence') / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
(OUTPUT_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2))
(OUTPUT_DIR / 'query_evidence.json').write_text(json.dumps(evidence, indent=2, default=str))
for variable_name in ('supporting_demo_rows', 'reconciliation', 'metrics', 'evaluation_results', 'record_trace'):
    result_frame = globals().get(variable_name)
    if isinstance(result_frame, pd.DataFrame):
        result_frame.to_json(OUTPUT_DIR / f'{variable_name}.json', orient='records', indent=2, default_handler=str)
if 'figure' in globals():
    figure.savefig(OUTPUT_DIR / 'reconciliation_notebook_output.png', dpi=180, bbox_inches='tight')
print('Evidence saved in notebook filesystem:', OUTPUT_DIR)
print('Download or explicitly persist these files before stopping the notebook runtime. No external publication performed.')

In [ ]:
if not APPROVE_CANCEL:
    raise RuntimeError('Cancellation disabled. Set APPROVE_CANCEL=True only to cancel an active evaluation.')
APPROVE_CANCEL = False
select_namespace()
execute_file('07_cancel_evaluation.sql')

In [ ]:
from IPython.display import Code, display
for relative in (
    'supporting_code/examples/live_verification_demo/financebench_retrieval.py',
    'supporting_code/examples/live_verification_demo/financebench_experiment.py',
    'supporting_code/examples/live_verification_demo/managed_mcp_probe.py',
    'managed_mcp.yaml',
):
    verified_skill_artifact(relative)
from examples.live_verification_demo.financebench_experiment import offline_demo, call_budget
from examples.live_verification_demo.managed_mcp_probe import probe

finance_fixture_results = offline_demo()
display(finance_fixture_results)
display(call_budget(2, group_size=4, repetitions=3))
display(Code(verified_skill_artifact('managed_mcp.yaml').read_text(), language='yaml'))

In [ ]:
import os
import uuid
APPROVE_CREATE_MANAGED_MCP = False
APPROVE_MANAGED_MCP_PROBE = False
APPROVED_MCP_HOST = ''  # Leave blank to derive from verified current organization/account.
_mcp_pat_secret = globals().get('_mcp_pat_secret', '')
_mcp_pat_name = globals().get('_mcp_pat_name', '')
_mcp_pat_user = globals().get('_mcp_pat_user', '')
mcp_evidence = globals().get('mcp_evidence', [])


def create_finview_pat():
    """Generate your own PAT outside this notebook; it is read from FINVIEW_MCP_PAT at runtime."""
    global _mcp_pat_secret, _mcp_pat_name, _mcp_pat_user
    _mcp_pat_secret = os.environ.get('FINVIEW_MCP_PAT', '<Generate your own PAT>')
    _mcp_pat_name = os.environ.get('FINVIEW_MCP_PAT_NAME', '<Your PAT name>')
    _mcp_pat_user = checked_context()['USER_NAME']
    if _mcp_pat_secret == '<Generate your own PAT>':
        raise ValueError('Generate your own PAT and supply it via FINVIEW_MCP_PAT; this notebook does not create tokens.')
    print('User-supplied PAT loaded; secret held in memory only.')


def ensure_managed_mcp():
    global APPROVE_CREATE_MANAGED_MCP
    checked_context()
    rows = execute_recorded(f'SHOW MCP SERVERS IN SCHEMA {namespace}', 'existing MCP servers')
    if not rows.empty and 'FINANCIAL_VERIFICATION_MCP' in set(rows['name']):
        return execute_recorded(f'DESCRIBE MCP SERVER {namespace}.FINANCIAL_VERIFICATION_MCP', 'reuse MCP server')
    if not APPROVE_CREATE_MANAGED_MCP:
        raise PermissionError('No MCP server exists; review 08_create_managed_mcp.sql then approve its creation')
    APPROVE_CREATE_MANAGED_MCP = False
    execute_file('08_create_managed_mcp.sql')


# Load your own PAT (see create_finview_pat); remove/rotate it yourself when finished.
create_finview_pat()

In [ ]:
# FINVIEW_CLIENT: calls REVIEW_RECONCILIATION directly via Snowpark session.
import time
import threading


class FinviewMCPClient:
    def __init__(self, max_calls=7):
        checked_context()
        self.function_name = f'{namespace}.REVIEW_RECONCILIATION'
        self.calls, self.max_calls, self.discovered = 0, max_calls, False
        self.lock = threading.Lock()

    def discover(self):
        self.discovered = False
        checked_context()
        rows = session.sql(f"DESCRIBE FUNCTION {self.function_name}(VARCHAR)").collect()
        if not rows:
            raise ValueError('Function not found')
        self.discovered = True
        return {'tools': [{'name': 'review_distribution_reconciliation',
                           'inputSchema': {'type': 'object', 'properties': {'P_CASE_ID': {'type': 'string'}}, 'required': ['P_CASE_ID']}}]}

    def review(self, case_id):
        if not self.discovered or case_id not in {'BALANCED', 'SHORTFALL', 'OVERALLOCATED', 'MISSING', 'INCOMPLETE', 'UNKNOWN'}:
            raise ValueError('Discover tools and use an allowed synthetic case')
        checked_context()
        with self.lock:
            if self.calls >= self.max_calls:
                raise RuntimeError('Call budget exhausted')
            self.calls += 1
        started = time.monotonic()
        event = {'method': 'tools/call', 'case_id': case_id, 'execution_kind': 'direct_sql',
                 'utc': datetime.now(timezone.utc).isoformat()}
        try:
            row = session.sql(f"SELECT {self.function_name}('{case_id}') AS REVIEW").collect()[0].as_dict()
            result = row['REVIEW']
            event.update(status='ok', latency_ms=(time.monotonic() - started) * 1000)
            mcp_evidence.append(event)
            return result
        except Exception as exc:
            event.update(status='error', error_type=type(exc).__name__, latency_ms=(time.monotonic() - started) * 1000)
            mcp_evidence.append(event)
            raise


print('Direct SQL client defined. No PAT or HTTP required.')

In [ ]:
# Local-only export; call explicitly after reviewing responses for sensitive data.
def export_financial_mcp_evidence(output_directory):
    from pathlib import Path
    import json
    directory = Path(output_directory)
    directory.mkdir(parents=True, exist_ok=False)
    (directory / 'synthetic_contract_results.json').write_text(
        json.dumps(finance_fixture_results, indent=2))
    (directory / 'mcp_protocol_evidence.json').write_text(
        json.dumps(mcp_evidence, indent=2))
    print('Saved local evidence. Empty MCP evidence means no captured MCP invocation.')

# Download/persist notebook runtime files before termination. Review before adding
# screenshots to the article. Do not label mock/unit-test evidence as live MCP.

In [ ]:
# SEC_AGENT_CONFIG: independent names avoid changing any earlier notebook state.
import json
import re
import hashlib
from datetime import date, datetime, timezone
from snowflake.snowpark.context import get_active_session

sec_session = get_active_session()
SEC_EXPECTED_ACCOUNT = '<your account>'
SEC_ROLE = '<your role>'
SEC_WAREHOUSE = '<your warehouse>'
SEC_DESTINATION = '<your db>.<your schema>'
SEC_SOURCE = 'SNOWFLAKE_PUBLIC_DATA.PUBLIC_DATA_FREE'
SEC_START = '2024-01-01'
SEC_END = '2026-10-01'  # Exclusive acceptance-time/filed-date cutoff; fixed demo window.
SEC_FISCAL_YEAR = 2024
APPROVE_SEC_DEPLOY = False
APPROVE_SEC_AGENT_CALLS = False
sec_events = globals().get('sec_events', [])
sec_created = globals().get('sec_created', [])
sec_call_attempts = globals().get('sec_call_attempts', set())
sec_preflight_hash = None
sec_validated_hash = None


def sec_identifier(value, parts=1):
    if not isinstance(value, str) or len(value.split('.')) != parts:
        raise ValueError('Unexpected identifier shape')
    if not all(re.fullmatch(r'[A-Z_][A-Z0-9_]*', part) for part in value.split('.')):
        raise ValueError('Use uppercase unquoted identifiers only')
    return value


def sec_digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, default=str).encode()).hexdigest()


def sec_context(require_role=True):
    row = sec_session.sql('SELECT CURRENT_ACCOUNT_NAME() AS ACCOUNT_NAME, CURRENT_ROLE() AS ROLE_NAME, CURRENT_WAREHOUSE() AS WAREHOUSE_NAME').collect()[0].as_dict()
    if row['ACCOUNT_NAME'] != SEC_EXPECTED_ACCOUNT:
        raise RuntimeError(f"Expected {SEC_EXPECTED_ACCOUNT}; active account is {row['ACCOUNT_NAME']}. No deployment allowed.")
    if require_role and (row['ROLE_NAME'] != SEC_ROLE or row['WAREHOUSE_NAME'] != SEC_WAREHOUSE):
        raise RuntimeError('Select the configured role and warehouse before proceeding')
    return row


def sec_execute(sql, label, params=None):
    sec_context()
    event = {'label': label, 'utc': datetime.now(timezone.utc).isoformat()}
    history = None
    try:
        with sec_session.query_history() as history:
            rows = sec_session.sql(sql, params=params).collect()
        event.update(status='ok', row_count=len(rows))
        return [row.as_dict() for row in rows]
    except Exception as exc:
        event.update(status='error', error_type=type(exc).__name__)
        raise
    finally:
        event['query_ids'] = [entry.query_id for entry in history.queries] if history is not None else []
        sec_events.append(event)


print('Independent SEC configuration loaded. No DDL, role switch or inference executed.')

In [ ]:
# SEC_AGENT_CONFIG: independent names avoid changing any earlier notebook state.
import json
import re
import hashlib
from datetime import date, datetime, timezone
from snowflake.snowpark.context import get_active_session

sec_session = get_active_session()
SEC_EXPECTED_ACCOUNT = '<your account>'
SEC_ROLE = '<your role>'
SEC_WAREHOUSE = '<your warehouse>'
SEC_DESTINATION = '<your db>.<your schema>'
SEC_SOURCE = 'SNOWFLAKE_PUBLIC_DATA.PUBLIC_DATA_FREE'
SEC_START = '2024-01-01'
SEC_END = '2026-10-01'  # Exclusive acceptance-time/filed-date cutoff; fixed demo window.
SEC_FISCAL_YEAR = 2024
APPROVE_SEC_DEPLOY = False
APPROVE_SEC_AGENT_CALLS = False
sec_events = globals().get('sec_events', [])
sec_created = globals().get('sec_created', [])
sec_call_attempts = globals().get('sec_call_attempts', set())
sec_preflight_hash = None
sec_validated_hash = None


def sec_identifier(value, parts=1):
    if not isinstance(value, str) or len(value.split('.')) != parts:
        raise ValueError('Unexpected identifier shape')
    if not all(re.fullmatch(r'[A-Z_][A-Z0-9_]*', part) for part in value.split('.')):
        raise ValueError('Use uppercase unquoted identifiers only')
    return value


def sec_digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, default=str).encode()).hexdigest()


def sec_context(require_role=True):
    row = sec_session.sql('SELECT CURRENT_ACCOUNT_NAME() AS ACCOUNT_NAME, CURRENT_ROLE() AS ROLE_NAME, CURRENT_WAREHOUSE() AS WAREHOUSE_NAME').collect()[0].as_dict()
    if row['ACCOUNT_NAME'] != SEC_EXPECTED_ACCOUNT:
        raise RuntimeError(f"Expected {SEC_EXPECTED_ACCOUNT}; active account is {row['ACCOUNT_NAME']}. No deployment allowed.")
    if require_role and (row['ROLE_NAME'] != SEC_ROLE or row['WAREHOUSE_NAME'] != SEC_WAREHOUSE):
        raise RuntimeError('Select the configured role and warehouse before proceeding')
    return row


def sec_execute(sql, label, params=None):
    sec_context()
    event = {'label': label, 'utc': datetime.now(timezone.utc).isoformat()}
    history = None
    try:
        with sec_session.query_history() as history:
            rows = sec_session.sql(sql, params=params).collect()
        event.update(status='ok', row_count=len(rows))
        return [row.as_dict() for row in rows]
    except Exception as exc:
        event.update(status='error', error_type=type(exc).__name__)
        raise
    finally:
        event['query_ids'] = [entry.query_id for entry in history.queries] if history is not None else []
        sec_events.append(event)


print('Independent SEC configuration loaded. No DDL, role switch or inference executed.')

In [ ]:
# SEC_AGENT_DEFINITIONS: query-backed semantic tables create no supporting tables/views.
def build_sec_definitions():
    sec_identifier(SEC_SOURCE, 2)
    sec_identifier(SEC_DESTINATION, 2)
    sec_identifier(SEC_ROLE)
    sec_identifier(SEC_WAREHOUSE)
    if type(SEC_FISCAL_YEAR) is not int or not 2000 <= SEC_FISCAL_YEAR <= 2100:
        raise ValueError('Set a valid fiscal year')
    if date.fromisoformat(SEC_START) >= date.fromisoformat(SEC_END):
        raise ValueError('Invalid SEC date window')
    issuers = "'0001158449','0000866787','0000898173','0000040987'"
    filing_sql = f"""SELECT CIK, COMPANY_NAME, ADSH, FORM_TYPE, FILED_DATE, TIMESTAMP_ACCEPTED,
        FISCAL_YEAR, FISCAL_PERIOD
      FROM {SEC_SOURCE}.SEC_REPORT_INDEX
      WHERE CIK IN ({issuers})
        AND FILED_DATE >= '{SEC_START}' AND FILED_DATE < '{SEC_END}'
        AND TIMESTAMP_ACCEPTED < '{SEC_END}'::TIMESTAMP_NTZ"""
    financial_sql = f"""WITH filings AS ({filing_sql}), eligible AS (
      SELECT m.CIK, m.COMPANY_NAME, m.ADSH, m.FISCAL_YEAR,
             m.PERIOD_START_DATE, m.PERIOD_END_DATE, m.TAG, m.VALUE,
             f.FILED_DATE, f.TIMESTAMP_ACCEPTED
      FROM {SEC_SOURCE}.SEC_METRICS_TIMESERIES m
      JOIN filings f ON m.CIK=f.CIK AND m.ADSH=f.ADSH
      WHERE m.CIK IN ({issuers}) AND m.FISCAL_YEAR={SEC_FISCAL_YEAR}
        AND m.FISCAL_PERIOD='FY' AND m.FREQUENCY='ANNUAL' AND m.UNIT='USD'
        AND f.FORM_TYPE IN ('10-K','10-K/A')
        AND m.PERIOD_START_DATE IS NOT NULL AND m.PERIOD_END_DATE IS NOT NULL
        AND DATEDIFF('day',m.PERIOD_START_DATE,m.PERIOD_END_DATE) BETWEEN 350 AND 380
        AND m.TAG IN ('RevenueFromContractWithCustomerExcludingAssessedTax','Revenues','OperatingIncomeLoss')
        AND NULLIF(TRIM(m.BUSINESS_SEGMENT),'') IS NULL
        AND NULLIF(TRIM(m.BUSINESS_SUBSEGMENT),'') IS NULL
        AND NULLIF(TRIM(m.GEO_NAME),'') IS NULL AND NULLIF(TRIM(m.GEO_ID),'') IS NULL
        AND NULLIF(TRIM(m.CUSTOMER),'') IS NULL AND NULLIF(TRIM(m.LEGAL_ENTITY),'') IS NULL
    ), latest AS (
      SELECT * FROM eligible
      QUALIFY DENSE_RANK() OVER (PARTITION BY CIK,FISCAL_YEAR,PERIOD_START_DATE,PERIOD_END_DATE
        ORDER BY TIMESTAMP_ACCEPTED DESC,ADSH DESC)=1
    ), grouped AS (
      SELECT CIK,COMPANY_NAME,FISCAL_YEAR,PERIOD_START_DATE,PERIOD_END_DATE,ADSH,FILED_DATE,
        TIMESTAMP_ACCEPTED,
        COUNT_IF(TAG != 'OperatingIncomeLoss') AS REVENUE_ROWS,
        COUNT(DISTINCT IFF(TAG != 'OperatingIncomeLoss',VALUE,NULL)) AS REVENUE_VALUES,
        COUNT_IF(TAG != 'OperatingIncomeLoss' AND (VALUE IS NULL OR VALUE IN ('NaN'::FLOAT,'inf'::FLOAT,'-inf'::FLOAT))) AS BAD_REVENUE,
        MAX(IFF(TAG != 'OperatingIncomeLoss',VALUE,NULL)) AS REVENUE_CANDIDATE,
        COUNT_IF(TAG = 'OperatingIncomeLoss') AS OPERATING_ROWS,
        COUNT(DISTINCT IFF(TAG = 'OperatingIncomeLoss',VALUE,NULL)) AS OPERATING_VALUES,
        COUNT_IF(TAG = 'OperatingIncomeLoss' AND (VALUE IS NULL OR VALUE IN ('NaN'::FLOAT,'inf'::FLOAT,'-inf'::FLOAT))) AS BAD_OPERATING,
        MAX(IFF(TAG = 'OperatingIncomeLoss',VALUE,NULL)) AS OPERATING_CANDIDATE
      FROM latest GROUP BY CIK,COMPANY_NAME,FISCAL_YEAR,PERIOD_START_DATE,PERIOD_END_DATE,ADSH,FILED_DATE,TIMESTAMP_ACCEPTED
    ), checked AS (
      SELECT *, IFF(REVENUE_VALUES=1 AND COALESCE(BAD_REVENUE,0)=0,REVENUE_CANDIDATE,NULL) AS REVENUE_USD,
        IFF(OPERATING_VALUES=1 AND COALESCE(BAD_OPERATING,0)=0,OPERATING_CANDIDATE,NULL) AS OPERATING_INCOME_USD
      FROM grouped
    ) SELECT CIK,COMPANY_NAME,FISCAL_YEAR,PERIOD_START_DATE,PERIOD_END_DATE,ADSH,FILED_DATE,TIMESTAMP_ACCEPTED,
        'USD' AS UNIT, REVENUE_USD,OPERATING_INCOME_USD,
        IFF(REVENUE_USD>0,100.0*OPERATING_INCOME_USD/NULLIF(REVENUE_USD,0),NULL) AS OPERATING_MARGIN_PCT,
        CASE WHEN REVENUE_VALUES>1 OR OPERATING_VALUES>1 THEN 'CONFLICTING_VALUES'
             WHEN REVENUE_USD IS NULL OR OPERATING_INCOME_USD IS NULL THEN 'MISSING_METRIC'
             WHEN REVENUE_USD<=0 THEN 'INVALID_DENOMINATOR' ELSE 'COMPARABLE_INPUTS' END AS CHECK_STATUS,
        REVENUE_ROWS,OPERATING_ROWS
      FROM checked"""
    def dimension(name, datatype, description):
        return {'name': name, 'expr': name, 'data_type': datatype, 'description': description}
    common = [dimension('CIK','VARCHAR','SEC issuer identifier; four auto-parts issuers only.'),
              dimension('COMPANY_NAME','VARCHAR','Issuer name reported in the source.'),
              dimension('ADSH','VARCHAR','Filing accession; preserve as provenance.')]
    financial_table = {
        'name': 'COMPS', 'description': 'Latest eligible filing per issuer, annual period and fiscal year within the fixed window; not historical provider vintages.',
        'base_table': {'definition': financial_sql},
        'dimensions': common + [dimension('FISCAL_YEAR','FLOAT','Fiscal label; periods differ across issuers.'),
            dimension('UNIT','VARCHAR','USD only.'), dimension('CHECK_STATUS','VARCHAR','Report missing/conflicting metrics, never replace with zero.')],
        'time_dimensions': [dimension(name,'DATE',description) for name,description in (
            ('PERIOD_START_DATE','Actual annual reporting start.'),('PERIOD_END_DATE','Actual annual reporting end.'),
            ('FILED_DATE','SEC filed date, not economic event date.'))],
        'facts': [dimension(name,'FLOAT',description) for name,description in (
            ('REVENUE_USD','One unambiguous consolidated revenue value, not sum of source labels.'),
            ('OPERATING_INCOME_USD','Operating income, not EBITDA. NULL if missing or conflicting.'),
            ('OPERATING_MARGIN_PCT','100 times operating income divided by positive revenue for the same filing and period. NULL if unavailable.'))],
    }
    filing_table = {'name':'FILINGS','description':'Dated filing inventory; no text evidence or event classifications.',
        'base_table':{'definition':filing_sql},
        'dimensions': common + [dimension('FORM_TYPE','VARCHAR','SEC form, e.g. 8-K, 10-K or 10-Q.'),
            dimension('FISCAL_YEAR','VARCHAR','Fiscal year label, possibly absent.'),dimension('FISCAL_PERIOD','VARCHAR','Fiscal period label.')],
        'time_dimensions':[dimension('FILED_DATE','DATE','SEC filed date; not event date.'),
                           dimension('TIMESTAMP_ACCEPTED','TIMESTAMP_NTZ','SEC acceptance time as provided; not provider availability time.')],
        'metrics':[{'name':'FILING_COUNT','expr':'COUNT(DISTINCT ADSH)','description':'Distinct filings; only use after accession uniqueness preflight.'}]}
    models = {
        'PE_PUBLIC_COMPS_SV': {'name':'PE_PUBLIC_COMPS_SV','description':'Four-issuer FY public-comparables review. Missing operating income is not zero.',
            'tables':[financial_table], 'custom_instructions':'Return case-level facts, not SUM across filings or periods. Always show CIK, company, fiscal year, actual period dates, accession, unit and check_status. Include NULL values and warnings. Do not compute EBITDA, WACC, valuation, or investment advice. Do not assume identical fiscal calendars. Latest eligible source filing is retrospective, not a point-in-time backtest.'},
        'ISSUER_FILINGS_SV': {'name':'ISSUER_FILINGS_SV','description':'Four-issuer dated SEC filing inventory, without filing text.',
            'tables':[filing_table], 'custom_instructions':'Show issuer, accession, form and filed date. Count distinct accession numbers. Distinguish SEC acceptance, filed date and event date; event dates and narrative contents are not available. Restrict requests to the configured date window and issuer set. Never infer acquisition or financing events from form type alone.'}}
    return {'models':models,'financial_sql':financial_sql,'filing_sql':filing_sql,
            'configuration':{'account':SEC_EXPECTED_ACCOUNT,'role':SEC_ROLE,'warehouse':SEC_WAREHOUSE,
                             'destination':SEC_DESTINATION,'source':SEC_SOURCE,'start':SEC_START,'end':SEC_END,'year':SEC_FISCAL_YEAR}}


sec_definitions = build_sec_definitions()
print(json.dumps(sec_definitions['models'], indent=2))  # JSON is also valid YAML; no extra parser package needed.

In [ ]:
# SEC_AGENT_PREFLIGHT: run before any creation. Read-only except selecting role/warehouse.
def preflight_sec_agents():
    global sec_preflight_hash
    sec_preflight_hash = None
    sec_context(require_role=False)  # Account checked BEFORE role/warehouse selection.
    sec_session.use_role(sec_identifier(SEC_ROLE))
    sec_session.use_warehouse(sec_identifier(SEC_WAREHOUSE))
    sec_context()
    sec_execute(f'DESCRIBE SCHEMA {sec_identifier(SEC_DESTINATION, 2)}', 'destination_exists')
    for table in ('SEC_METRICS_TIMESERIES','SEC_REPORT_INDEX'):
        sec_execute(f'DESCRIBE TABLE {sec_identifier(SEC_SOURCE, 2)}.{table}', 'source_schema_' + table)
    definitions = build_sec_definitions()
    if definitions != sec_definitions:
        raise RuntimeError('Configuration changed; rerun definitions and review before preflight')
    index_sql = definitions['filing_sql']
    invalid = sec_execute(f'''SELECT COUNT(*) AS BAD_KEYS FROM (
        SELECT CIK,ADSH FROM ({index_sql}) GROUP BY CIK,ADSH
        HAVING COUNT(*) != 1 OR CIK IS NULL OR ADSH IS NULL)''', 'filing_key_check')
    if invalid[0]['BAD_KEYS'] != 0:
        raise RuntimeError('Filing keys are missing or duplicated; do not deploy an ambiguous model')
    grain_check = sec_execute(f'''SELECT COUNT(*) AS BAD_GRAINS FROM (
        SELECT CIK,FISCAL_YEAR,PERIOD_START_DATE,PERIOD_END_DATE
        FROM ({definitions['financial_sql']})
        GROUP BY CIK,FISCAL_YEAR,PERIOD_START_DATE,PERIOD_END_DATE HAVING COUNT(*) != 1)''', 'financial_grain_check')
    if grain_check[0]['BAD_GRAINS'] != 0:
        raise RuntimeError('Financial reporting grain is ambiguous')
    financial = sec_execute(definitions['financial_sql'] + ' ORDER BY CIK,PERIOD_END_DATE LIMIT 20', 'financial_preview')
    inventory = sec_execute(f'''SELECT CIK,COUNT(DISTINCT ADSH) AS FILING_COUNT,
        MIN(FILED_DATE) AS FIRST_FILED, MAX(FILED_DATE) AS LAST_FILED
        FROM ({index_sql}) GROUP BY CIK ORDER BY CIK''', 'filing_preview')
    expected_issuers = {'0001158449','0000866787','0000898173','0000040987'}
    if {row['CIK'] for row in financial} != expected_issuers or {row['CIK'] for row in inventory} != expected_issuers:
        raise RuntimeError('The selected window lacks coverage for one or more demo issuers; inspect source/window')
    keys = [(row['CIK'], row['FISCAL_YEAR'], row['PERIOD_START_DATE'], row['PERIOD_END_DATE']) for row in financial]
    if len(set(keys)) != len(keys):
        raise RuntimeError('Financial reporting grain is ambiguous')
    for name, model in definitions['models'].items():
        sec_execute('CALL SYSTEM$CREATE_SEMANTIC_VIEW_FROM_YAML(?, ?, TRUE)',
                    'validate_' + name, [SEC_DESTINATION, json.dumps(model)])
    sec_preflight_hash = sec_digest(definitions)
    print(json.dumps({'financial_preview':financial,'filing_preview':inventory,
                      'preflight_hash':sec_preflight_hash}, indent=2, default=str))
    return {'financial': financial, 'filings': inventory}


sec_source_preview = preflight_sec_agents()

In [ ]:
# SEC_AGENT_SPECS: two T01 single-domain Analyst agents; no generic SQL/MCP/write tools.
def build_sec_agent_specs():
    definitions = build_sec_definitions()
    questions = {
        'PE_DILIGENCE_AGENT': f'Compare FY{SEC_FISCAL_YEAR} revenue and operating margins for AAP, AutoZone, O Reilly and Genuine Parts. Include actual reporting periods, accession numbers and missing-metric warnings.',
        'CAPITAL_MARKETS_DISCLOSURE_AGENT': f'Count 8-K filings by issuer from {SEC_START} inclusive to {SEC_END} exclusive for AAP, AutoZone, O Reilly and Genuine Parts; show the first and last filing dates.'}
    routing = {
        'PE_DILIGENCE_AGENT': ('public_comps', 'PE_PUBLIC_COMPS_SV',
            'Annual public-company financial comparisons with explicit units, filing provenance and missing-value status.',
            'Always query public_comps. Use the semantic facts at issuer/fiscal-year/period/accession grain. Never sum duplicated filings or average margins across issuers. Show actual period dates and disclose differing fiscal calendars. NULL operating income means margin unavailable, not zero. These are retrospective public comps, not private-company EBITDA, valuation or investment advice.'),
        'CAPITAL_MARKETS_DISCLOSURE_AGENT': ('issuer_filings', 'ISSUER_FILINGS_SV',
            'SEC filing inventories and distinct accession counts for four auto-parts issuers in a fixed date window.',
            'Always query issuer_filings. Cite company, CIK, accession, form and filed date for listings. Count distinct accessions for inventories. This tool has no document text: do not infer acquisitions, financing events or event dates from form type alone. Ask for source text if those are requested. Never claim settlement reconciliation or trade approval.')}
    specs = {}
    for name, (tool, semantic_view, description, instructions) in routing.items():
        specs[name] = {
            'models': {'orchestration':'auto'},
            'orchestration': {'budget':{'seconds':60,'tokens':8000}, 'tool_not_accessible':'reject'},
            'instructions': {'response':'Be concise. Cite returned evidence, show missing/conflicting data and limitations. No investment, payment or trade authorization.',
                             'orchestration':instructions,
                             'sample_questions':[{'question':questions[name]}]},
            'tools':[{'tool_spec':{'type':'cortex_analyst_text_to_sql','name':tool,'description':description}}],
            'tool_resources':{tool:{'semantic_view':SEC_DESTINATION + '.' + semantic_view,
                                   'execution_environment':{'type':'warehouse','warehouse':SEC_WAREHOUSE,'query_timeout':60}}}}
    return specs, questions


sec_agent_specs, sec_questions = build_sec_agent_specs()
sec_review = {'definitions':sec_definitions,'agents':sec_agent_specs}
sec_review_hash = sec_digest(sec_review)
print(json.dumps({'agents':sec_agent_specs,'review_hash':sec_review_hash}, indent=2))

In [ ]:
# SEC_AGENT_DEPLOY: plain CREATE only; no replace, grants or source writes.
def sec_literal(value):
    return "'" + value.replace("'", "''") + "'"


def sec_semantic_ddl(model):
    table = model['tables'][0]
    alias = sec_identifier(table['name'])
    clauses = [f"CREATE SEMANTIC VIEW {SEC_DESTINATION}.{sec_identifier(model['name'])}",
               f"TABLES ({alias} AS ({table['base_table']['definition']}))"]
    for keyword, fields in (('FACTS',table.get('facts',[])),
                            ('DIMENSIONS',table.get('dimensions',[]) + table.get('time_dimensions',[])),
                            ('METRICS',table.get('metrics',[]))):
        if fields:
            expressions = [f"{alias}.{sec_identifier(field['name'])} AS {field['expr']} COMMENT = {sec_literal(field['description'])}"
                           for field in fields]
            clauses.append(keyword + ' (' + ',\n'.join(expressions) + ')')
    clauses += ['COMMENT = ' + sec_literal(model['description']),
                'AI_SQL_GENERATION ' + sec_literal(model['custom_instructions'])]
    return '\n'.join(clauses)


def sec_assert_review():
    sec_context()
    current = {'definitions':build_sec_definitions(),'agents':build_sec_agent_specs()[0]}
    if (sec_digest(current) != sec_review_hash or sec_digest(current['definitions']) != sec_preflight_hash
            or current != {'definitions':sec_definitions,'agents':sec_agent_specs}
            or sec_questions != build_sec_agent_specs()[1]):
        raise RuntimeError('Definitions/configuration changed or preflight missing; review and preflight again')


def deploy_sec_agents():
    global APPROVE_SEC_DEPLOY
    if not APPROVE_SEC_DEPLOY:
        raise PermissionError('Review the four definitions, then explicitly approve deployment')
    APPROVE_SEC_DEPLOY = False
    sec_assert_review()
    if sec_created:
        raise RuntimeError('Prior partial/full creation recorded. Inspect existing objects; no automatic replacement.')
    for kind, names in (('SEMANTIC VIEWS',sec_definitions['models']),('AGENTS',sec_agent_specs)):
        existing = sec_execute(f'SHOW {kind} IN SCHEMA {SEC_DESTINATION}', 'collision_check')
        visible_names = {str(row.get('name',row.get('NAME',''))).upper() for row in existing}
        if visible_names.intersection(names):
            raise RuntimeError('One or more target objects already exist; stop instead of replacing')
    for name, model in sec_definitions['models'].items():
        sec_execute(sec_semantic_ddl(model), 'create_' + name)
        sec_created.append(SEC_DESTINATION + '.' + name)
    for name, spec in sec_agent_specs.items():
        # Embed the reviewed, tracked specification; no account-specific credentials.
        statement = f'CREATE AGENT {SEC_DESTINATION}.{sec_identifier(name)} FROM SPECIFICATION ' + sec_literal(json.dumps(spec))
        sec_execute(statement, 'create_' + name)
        sec_created.append(SEC_DESTINATION + '.' + name)
    return list(sec_created)


for model in sec_definitions['models'].values():
    print(sec_semantic_ddl(model) + ';\n')
print('After reviewing, run APPROVE_SEC_DEPLOY = True; deploy_sec_agents() in a separate cell. DDL is not transactional.')

In [ ]:
# SEC_AGENT_CALLS: exactly one question per agent, no automatic retries or token extraction.
sec_agent_results = globals().get('sec_agent_results', {})


def run_sec_agent_examples():
    global APPROVE_SEC_AGENT_CALLS
    if not APPROVE_SEC_AGENT_CALLS:
        raise PermissionError('Approve two billable Cortex Agent calls before running')
    APPROVE_SEC_AGENT_CALLS = False
    sec_assert_review()
    if sec_validated_hash != sec_review_hash:
        raise RuntimeError('Validate semantic/source parity before running agent questions')
    if sec_call_attempts:
        raise RuntimeError('An example call was already attempted. Review saved results/errors before any new run.')
    for name, question in sec_questions.items():
        sec_call_attempts.add(name)
        request = {'stream':False, 'messages':[{'role':'user','content':[{'type':'text','text':question}]}],
                   'tool_choice':{'type':'required'}}
        rows = sec_execute('SELECT SNOWFLAKE.CORTEX.DATA_AGENT_RUN(?, ?) AS RESPONSE',
                           'invoke_' + name, [SEC_DESTINATION + '.' + name, json.dumps(request)])
        response = rows[0]['RESPONSE']
   

In [ ]:
# SEC_AGENT_EXERCISE: objects validated; now run agent calls.
APPROVE_SEC_AGENT_CALLS = True
sec_call_attempts.clear()
sec_validated_hash = sec_review_hash
run_sec_agent_examples()

In [ ]:
# FINVIEW_TRULENS_CONFIG: definitions only; no calls or registration.
import os, hashlib, uuid, time
import pandas as pd
from datetime import datetime, timezone
APPROVE_TRULENS_RUN = True
APPROVE_JUDGE_METRICS = True
JUDGE_MODEL = 'llama3.1-70b'  # Choose an allowed Cortex judge model explicitly.
finview_eval_attempted = globals().get('finview_eval_attempted', False)
finview_metrics_attempted = globals().get('finview_metrics_attempted', False)


def run_finview_mcp_evaluation():
    global APPROVE_TRULENS_RUN, finview_eval_attempted, finview_run, finview_tru_app
    if not APPROVE_TRULENS_RUN:
        raise PermissionError('Approve registration and seven MCP requests for the six-case evaluation')
    APPROVE_TRULENS_RUN = False
    checked_context()
    select_namespace()
    if finview_eval_attempted:
        raise RuntimeError('Evaluation already attempted; poll status instead of rerunning')
    if not JUDGE_MODEL:
        raise ValueError('Set JUDGE_MODEL before starting')
    os.environ['TRULENS_OTEL_TRACING'] = '1'
    from trulens.apps.app import TruApp
    from trulens.connectors.snowflake import SnowflakeConnector
    from trulens.core import TruSession
    from trulens.core.run import RunConfig
    from trulens.core.otel.instrument import instrument
    from trulens.otel.semconv.trace import SpanAttributes
    client = FinviewMCPClient(max_calls=7)
    finview_eval_attempted = True
    client.discover()

    class FinancialMCPApplication:
        @instrument(span_type=SpanAttributes.SpanType.RETRIEVAL,
                    attributes={SpanAttributes.RETRIEVAL.QUERY_TEXT:'query',SpanAttributes.RETRIEVAL.RETRIEVED_CONTEXTS:'return'})
        def retrieve(self, query: str, gold: str = None) -> list:
            return [client.review(query)]

        @instrument(attributes={SpanAttributes.RECORD_ROOT.INPUT:'query',SpanAttributes.RECORD_ROOT.OUTPUT:'return'})
        def respond(self, query: str, gold: str = None) -> str:
            return self.retrieve(query)[0]

    evaluation_df = pd.DataFrame([
        {'query':'BALANCED','gold':'USD proceeds 300000000; allocated_total 300000000; gap 0; checks_passed; payment_authorized false; synthetic.'},
        {'query':'SHORTFALL','gold':'USD proceeds 300000000; allocated_total 237000000; gap 63000000; blocked_reconciliation; payment_authorized false; synthetic.'},
        {'query':'OVERALLOCATED','gold':'USD proceeds 300000000; allocated_total 301000000; gap -1000000; blocked_reconciliation; payment_authorized false; synthetic.'},
        {'query':'MISSING','gold':'blocked_incomplete; allocated_total null; gap null; zero actual allocations; payment_authorized false; synthetic.'},
        {'query':'INCOMPLETE','gold':'blocked_incomplete; allocated_total null; gap null; one null amount; payment_authorized false; synthetic.'},
        {'query':'UNKNOWN','gold':'blocked_unknown_case; payment_authorized false; synthetic.'},
    ])
    dataset_hash = hashlib.sha256(evaluation_df.to_json(orient='records').encode()).hexdigest()
    connector = SnowflakeConnector(snowpark_session=session, use_account_event_table=True)
    tru_session = TruSession(connector=connector)
    application = FinancialMCPApplication()
    finview_tru_app = TruApp(application,app_name='FINANCIAL_MCP_CLIENT_EVAL',app_version='finview_pat_v2',
                            connector=connector,main_method=application.respond)
    finview_run = finview_tru_app.add_run(run_config=RunConfig(
        run_name='MCP_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:8],
        description='Six synthetic PAT MCP cases; dataset sha256 ' + dataset_hash,
        label='managed-mcp-synthetic-v2',source_type='DATAFRAME',dataset_name='synthetic_reconciliation_v1',
        dataset_spec={'RECORD_ROOT.INPUT':'query','RETRIEVAL.QUERY_TEXT':'query','RECORD_ROOT.GROUND_TRUTH_OUTPUT':'gold'},
        llm_judge_name=JUDGE_MODEL))
    try:
        finview_run.start(input_df=evaluation_df)
    finally:
        from opentelemetry import trace
        provider = trace.get_tracer_provider()
        if hasattr(provider,'force_flush'):
            provider.force_flush()
    for _ in range(24):
        status = finview_run.get_status()
        status_name = getattr(status,'name',str(status).split('.')[-1])
        if status_name in {'INVOCATION_COMPLETED','INVOCATION_PARTIALLY_COMPLETED'} or 'FAILED' in status_name:
            break
        time.sleep(5)
    print(finview_run.get_status())
    return finview_run.describe()


def submit_finview_metrics():
    global APPROVE_JUDGE_METRICS, finview_metrics_attempted
    if not APPROVE_JUDGE_METRICS:
        raise PermissionError('Approve Cortex judge costs first')
    APPROVE_JUDGE_METRICS = False
    checked_context()
    if 'finview_run' not in globals():
        raise RuntimeError('Start and inspect the MCP evaluation first')
    status = finview_run.get_status()
    if getattr(status,'name',str(status).split('.')[-1]) != 'INVOCATION_COMPLETED':
        raise RuntimeError('All six invocations must complete before computing metrics')
    if finview_metrics_attempted:
        raise RuntimeError('Metrics already attempted; inspect results before retrying')
    finview_metrics_attempted = True
    return finview_run.compute_metrics(metrics=['correctness','groundedness'])


print('Set APPROVE_TRULENS_RUN=True and call run_finview_mcp_evaluation(); approve judge metrics separately.')

In [ ]:
APPROVE_TRULENS_RUN=True

In [ ]:
!pip install 'sqlalchemy>=2.0.0,<2.1.0' --quiet
import sqlalchemy; print('SQLAlchemy', sqlalchemy.__version__)
import snowflake.sqlalchemy; print('snowflake-sqlalchemy OK')
from trulens.connectors.snowflake import SnowflakeConnector; print('SnowflakeConnector OK')

In [ ]:
import json
print(f'Total MCP evidence entries: {len(mcp_evidence)}')
for e in mcp_evidence[-3:]:
    print(json.dumps({k:v for k,v in e.items() if k not in ('response',)}, indent=2, default=str))
print()
print('PAT secret available:', bool(_mcp_pat_secret), 'len:', len(_mcp_pat_secret) if _mcp_pat_secret else 0)

In [ ]:
print('_mcp_pat_name:', _mcp_pat_name)
print('_mcp_pat_secret set:', bool(_mcp_pat_secret))
print('_mcp_pat_secret length:', len(_mcp_pat_secret) if _mcp_pat_secret else 0)
print('ROLE:', ROLE)

In [ ]:
finview_eval_attempted = False
APPROVE_TRULENS_RUN = True
run_finview_mcp_evaluation()

In [ ]:
# Exercise the direct SQL client for all five synthetic cases
client = FinviewMCPClient(max_calls=7)
client.discover()
for case in ('BALANCED', 'SHORTFALL', 'OVERALLOCATED', 'MISSING', 'INCOMPLETE'):
    result = client.review(case)
    parsed = json.loads(result)
    print(f"{case:16s} | status={parsed['status']:25s} | gap={str(parsed.get('gap','N/A')):>12s} | payment_authorized={parsed['payment_authorized']}")
print(f'\n{client.calls} tool calls made. Evidence captured in mcp_evidence ({len(mcp_evidence)} entries).')

In [ ]:
# Display SEC agent call results from sec_events
for event in sec_events:
    if event['label'].startswith('invoke_'):
        print(f"\n{'='*80}")
        print(f"Agent: {event['label']}")
        print(f"Status: {event['status']}")
        print(f"Query IDs: {event.get('query_ids', [])}")
        print(f"Row count: {event.get('row_count', 'N/A')}")
print(f'\nTotal SEC events: {len(sec_events)}')

In [ ]:
# Manual evaluation: run all agents against their inputs, compare to ground truth.
import json
from IPython.display import display, Markdown

def extract_agent_text(resp_str):
    parsed = json.loads(resp_str)
    texts = []
    # DATA_AGENT_RUN returns {content: [{type: 'text', text: ...}, {type: 'tool_use', ...}, ...]}
    for item in parsed.get('content', []):
        if item.get('type') == 'text':
            texts.append(item['text'])
        elif item.get('type') == 'tool_result':
            for sub in item.get('tool_result', {}).get('content', []):
                if isinstance(sub, dict) and sub.get('type') == 'text':
                    texts.append(f"[tool:{sub['text'][:150]}]")
    # Also try choices->delta->content (alternate format)
    for choice in parsed.get('choices', []):
        for c in choice.get('delta', {}).get('content', []):
            if c.get('type') == 'text':
                texts.append(c['text'])
    return '\n'.join(texts) if texts else f'[NO_TEXT] keys={list(parsed.keys())}'

# --- WATERFALL_REVIEW_AGENT: 6 eval inputs ---
waterfall_inputs = session.sql('SELECT INPUT_QUERY, GROUND_TRUTH FROM <your db>.<your schema>.WATERFALL_EVAL_INPUT').collect()
waterfall_results = []
for row in waterfall_inputs:
    query = row['INPUT_QUERY']
    gt = json.loads(row['GROUND_TRUTH'])['ground_truth_output']
    request = json.dumps({'stream': False, 'messages': [{'role': 'user', 'content': [{'type': 'text', 'text': query}]}], 'tool_choice': {'type': 'required'}})
    resp = session.sql('SELECT SNOWFLAKE.CORTEX.DATA_AGENT_RUN(?, ?) AS R',
                       params=['<your db>.<your schema>.WATERFALL_REVIEW_AGENT', request]).collect()[0]['R']
    answer = extract_agent_text(resp)
    waterfall_results.append({'query': query, 'ground_truth': gt, 'answer': answer})

print('=== WATERFALL_REVIEW_AGENT (6 inputs) ===')
for i, r in enumerate(waterfall_results, 1):
    print(f'\n--- Input {i} ---')
    print(f'Q: {r["query"][:120]}')
    print(f'GT: {r["ground_truth"][:200]}')
    print(f'A:  {r["answer"][:500]}')
    print()

In [ ]:
# --- SEC AGENTS: PE_DILIGENCE_AGENT and CAPITAL_MARKETS_DISCLOSURE_AGENT ---
sec_eval_inputs = {
    'PE_DILIGENCE_AGENT': f'Compare FY{SEC_FISCAL_YEAR} revenue and operating margins for AAP, AutoZone, O Reilly and Genuine Parts. Include actual reporting periods, accession numbers and missing-metric warnings.',
    'CAPITAL_MARKETS_DISCLOSURE_AGENT': f'Count 8-K filings by issuer from {SEC_START} inclusive to {SEC_END} exclusive for AAP, AutoZone, O Reilly and Genuine Parts; show the first and last filing dates.'
}

sec_eval_results = {}
for agent_name, question in sec_eval_inputs.items():
    fqn = f'<your db>.<your schema>.{agent_name}'
    request = json.dumps({'stream': False, 'messages': [{'role': 'user', 'content': [{'type': 'text', 'text': question}]}], 'tool_choice': {'type': 'required'}})
    resp = session.sql('SELECT SNOWFLAKE.CORTEX.DATA_AGENT_RUN(?, ?) AS R', params=[fqn, request]).collect()[0]['R']
    answer = extract_agent_text(resp)
    sec_eval_results[agent_name] = {'question': question, 'answer': answer}

for agent_name, r in sec_eval_results.items():
    print(f'\n{"="*80}')
    print(f'=== {agent_name} ===')
    print(f'Q: {r["question"]}')
    print(f'\nA: {r["answer"][:2000]}')
    print()

In [ ]:
# Drop existing SEC objects for redeployment with updated budgets
for obj_type, name in [
    ('AGENT', 'CAPITAL_MARKETS_DISCLOSURE_AGENT'),
    ('AGENT', 'PE_DILIGENCE_AGENT'),
    ('SEMANTIC VIEW', 'ISSUER_FILINGS_SV'),
    ('SEMANTIC VIEW', 'PE_PUBLIC_COMPS_SV'),
]:
    session.sql(f'DROP {obj_type} IF EXISTS {namespace}.{name}').collect()
    print(f'Dropped {obj_type} {name}')
sec_created.clear()
print('Ready for redeployment.')

In [ ]:
# Redeploy ALL agents with budget 180s/16000 tokens
checked_context()

# 1. Recreate WATERFALL_REVIEW_AGENT (already dropped by prior run)
spec['orchestration']['budget'] = {'seconds': 180, 'tokens': 16000}
new_spec = json.dumps(spec).replace("'", "''")
session.sql(f"CREATE AGENT {namespace}.WATERFALL_REVIEW_AGENT FROM SPECIFICATION '{new_spec}'").collect()
print(f'WATERFALL_REVIEW_AGENT created with budget: 180s/16000')

# 2. Recreate SEC semantic views and agents
for name, model in sec_definitions['models'].items():
    sec_execute(sec_semantic_ddl(model), 'recreate_' + name)
    print(f'Created semantic view {name}')

for name, agent_spec in sec_agent_specs.items():
    agent_spec['orchestration']['budget'] = {'seconds': 180, 'tokens': 16000}
    spec_json = json.dumps(agent_spec).replace(chr(39), chr(39)+chr(39))
    sec_execute(f"CREATE AGENT {SEC_DESTINATION}.{sec_identifier(name)} FROM SPECIFICATION '{spec_json}'", 'recreate_' + name)
    print(f'Created agent {name} with budget: 180s/16000')

print('\nAll agents redeployed with 180s/16000 token budget.')

In [ ]:
# Chart: Waterfall Reconciliation Overview (interactive)
import plotly.graph_objects as go
from plotly.subplots import make_subplots

recon = session.sql(f'''
    SELECT CASE_ID, PROCEEDS, ALLOCATED_TOTAL, GAP, STATUS
    FROM {namespace}.WATERFALL_RECONCILIATION ORDER BY CASE_ID
''').collect()

cases = [r['CASE_ID'] for r in recon]
proceeds = [float(r['PROCEEDS'] or 0) / 1e6 for r in recon]
allocated = [float(r['ALLOCATED_TOTAL'] or 0) / 1e6 for r in recon]
gaps = [float(r['GAP'] or 0) / 1e6 for r in recon]
statuses = [r['STATUS'] for r in recon]

color_map = {'checks_passed': '#2ecc71', 'blocked_reconciliation': '#e74c3c', 'blocked_incomplete': '#f39c12'}
bar_colors = [color_map.get(s, '#95a5a6') for s in statuses]

fig = make_subplots(rows=1, cols=2, subplot_titles=('Proceeds vs Allocated', 'Reconciliation Gap'),
                    horizontal_spacing=0.12)

fig.add_trace(go.Bar(name='Proceeds', x=cases, y=proceeds, marker_color='#3498db',
                     hovertemplate='%{x}<br>Proceeds: $%{y:,.0f}M<extra></extra>'), row=1, col=1)
fig.add_trace(go.Bar(name='Allocated', x=cases, y=allocated, marker_color=bar_colors,
                     hovertemplate='%{x}<br>Allocated: $%{y:,.0f}M<extra></extra>'), row=1, col=1)

fig.add_trace(go.Bar(name='Gap', y=cases, x=gaps, orientation='h', marker_color=bar_colors,
                     text=[f'{s.replace("blocked_","").replace("checks_","")}' for s in statuses],
                     textposition='outside', textfont_size=10,
                     hovertemplate='%{y}<br>Gap: $%{x:,.0f}M<extra></extra>'), row=1, col=2)

fig.update_layout(height=450, title_text='Waterfall Reconciliation Overview',
                  barmode='group', showlegend=True, template='plotly_white')
fig.update_yaxes(title_text='USD (millions)', row=1, col=1)
fig.update_xaxes(title_text='Gap USD (millions)', row=1, col=2)
fig.show()

In [ ]:
# Chart: SEC Filing Counts & Financial Comparables (interactive)
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from collections import OrderedDict

filing_data = session.sql(f'''
    SELECT CIK, COMPANY_NAME, FORM_TYPE, COUNT(DISTINCT ADSH) AS CNT
    FROM {SEC_SOURCE}.SEC_REPORT_INDEX
    WHERE CIK IN ('0001158449','0000866787','0000898173','0000040987')
      AND FILED_DATE >= '{SEC_START}' AND FILED_DATE < '{SEC_END}'
    GROUP BY CIK, COMPANY_NAME, FORM_TYPE
    ORDER BY COMPANY_NAME, FORM_TYPE
''').collect()

companies = OrderedDict()
form_types = set()
for r in filing_data:
    name = r['COMPANY_NAME']
    ft = r['FORM_TYPE']
    form_types.add(ft)
    if name not in companies:
        companies[name] = {}
    companies[name][ft] = int(r['CNT'])

ft_totals = {ft: sum(c.get(ft, 0) for c in companies.values()) for ft in form_types}
top_forms = sorted(ft_totals, key=ft_totals.get, reverse=True)[:8]
comp_names = sorted(companies.keys())

fig = make_subplots(rows=1, cols=2, subplot_titles=(
    f'SEC Filings by Form Type<br>({SEC_START} to {SEC_END})',
    f'FY{SEC_FISCAL_YEAR} Revenue (Latest 10-K)'),
    horizontal_spacing=0.15)

colors = ['#66c2a5','#fc8d62','#8da0cb','#e78ac3','#a6d854','#ffd92f','#e5c494','#b3b3b3']
for i, ft in enumerate(top_forms):
    vals = [companies[c].get(ft, 0) for c in comp_names]
    fig.add_trace(go.Bar(name=ft, y=comp_names, x=vals, orientation='h',
                         marker_color=colors[i % len(colors)],
                         hovertemplate=f'{ft}<br>' + '%{y}: %{x} filings<extra></extra>'), row=1, col=1)

fin_data = session.sql(f'''
    SELECT m.CIK, m.COMPANY_NAME, m.VALUE AS REVENUE
    FROM {SEC_SOURCE}.SEC_METRICS_TIMESERIES m
    JOIN {SEC_SOURCE}.SEC_REPORT_INDEX r ON m.CIK=r.CIK AND m.ADSH=r.ADSH
    WHERE m.CIK IN ('0001158449','0000866787','0000898173','0000040987')
      AND m.FISCAL_YEAR = {SEC_FISCAL_YEAR} AND m.FISCAL_PERIOD = 'FY'
      AND m.FREQUENCY = 'ANNUAL' AND m.UNIT = 'USD'
      AND m.TAG IN ('RevenueFromContractWithCustomerExcludingAssessedTax','Revenues')
      AND r.FORM_TYPE IN ('10-K','10-K/A')
      AND NULLIF(TRIM(m.BUSINESS_SEGMENT),'') IS NULL
      AND NULLIF(TRIM(m.GEO_NAME),'') IS NULL
    QUALIFY ROW_NUMBER() OVER (PARTITION BY m.CIK ORDER BY r.FILED_DATE DESC) = 1
''').collect()

if fin_data:
    names = [r['COMPANY_NAME'] for r in fin_data]
    revs = [float(r['REVENUE'] or 0) / 1e9 for r in fin_data]
    fig.add_trace(go.Bar(name='Revenue', y=names, x=revs, orientation='h',
                         marker_color='#3498db', showlegend=False,
                         text=[f'${v:,.1f}B' for v in revs], textposition='outside',
                         hovertemplate='%{y}<br>Revenue: $%{x:,.2f}B<extra></extra>'), row=1, col=2)

fig.update_layout(height=450, barmode='stack', template='plotly_white',
                  title_text='SEC Issuer Overview: Filings & Revenue')
fig.update_xaxes(title_text='Distinct Filings', row=1, col=1)
fig.update_xaxes(title_text='Revenue (USD billions)', row=1, col=2)
fig.show()

In [ ]:
# Evaluation Charts: Agent Performance & Tool Call Evidence
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- 1. Direct tool call results (REVIEW_RECONCILIATION) ---
tool_cases = ['BALANCED', 'SHORTFALL', 'OVERALLOCATED', 'MISSING', 'INCOMPLETE']
tool_data = []
for case in tool_cases:
    row = session.sql(f"SELECT {namespace}.REVIEW_RECONCILIATION('{case}') AS R").collect()[0]
    parsed = json.loads(row['R'])
    tool_data.append(parsed)

proceeds = [float(d.get('proceeds', 0)) / 1e6 for d in tool_data]
allocated = [float(d.get('allocated_total', 0) if d.get('allocated_total') is not None else 0) / 1e6 for d in tool_data]
gap_vals = [float(d.get('gap', 0) if d.get('gap') is not None else 0) / 1e6 for d in tool_data]
status_list = [d.get('status', 'unknown') for d in tool_data]
payment = [d.get('payment_authorized', False) for d in tool_data]

status_colors = {'checks_passed': '#2ecc71', 'blocked_reconciliation': '#e74c3c',
                 'blocked_incomplete': '#f39c12', 'unknown': '#95a5a6'}

# --- 2. Agent orchestration outcomes ---
agent_eval = [
    ('WATERFALL: BALANCED', 'time_limit'),
    ('WATERFALL: SHORTFALL', 'time_limit'),
    ('WATERFALL: OVERALLOCATED', 'time_limit'),
    ('WATERFALL: MISSING', 'time_limit'),
    ('WATERFALL: INCOMPLETE', 'loop_detected'),
    ('WATERFALL: Adversarial', 'loop_detected'),
    ('PE_DILIGENCE', 'time_limit'),
    ('CAPITAL_MARKETS', 'loop_detected'),
]
eval_names = [e[0] for e in agent_eval]
eval_outcomes = [e[1] for e in agent_eval]
outcome_colors = {'time_limit': '#e67e22', 'loop_detected': '#e74c3c',
                  'completed': '#2ecc71', 'no_text': '#3498db'}
eval_bar_colors = [outcome_colors.get(o, '#95a5a6') for o in eval_outcomes]

# --- 3. SEC events summary ---
sec_event_labels = [e['label'] for e in sec_events if e.get('status') == 'ok']
sec_event_errors = [e['label'] for e in sec_events if e.get('status') == 'error']

# Build figure
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Tool Call Evidence: Reconciliation Status',
        'Tool Call Evidence: Gap Analysis',
        'Agent Orchestration Outcomes<br>(60s→180s budget)',
        'SEC Agent Event Log'
    ),
    vertical_spacing=0.15, horizontal_spacing=0.12
)

# Top-left: Status by case
fig.add_trace(go.Bar(
    x=tool_cases, y=[1]*5, marker_color=[status_colors[s] for s in status_list],
    text=status_list, textposition='inside', textfont_size=10,
    hovertemplate='%{x}<br>Status: %{text}<extra></extra>',
    showlegend=False
), row=1, col=1)
fig.update_yaxes(visible=False, row=1, col=1)

# Top-right: Gap values
fig.add_trace(go.Bar(
    x=tool_cases, y=gap_vals, marker_color=[status_colors[s] for s in status_list],
    text=[f'${g:,.0f}M' if g != 0 else 'N/A' for g in gap_vals],
    textposition='outside', textfont_size=9,
    hovertemplate='%{x}<br>Gap: $%{y:,.0f}M<extra></extra>',
    showlegend=False
), row=1, col=2)
fig.update_yaxes(title_text='Gap (USD millions)', row=1, col=2)

# Bottom-left: Agent orchestration outcomes
fig.add_trace(go.Bar(
    y=eval_names, x=[1]*len(eval_names), orientation='h',
    marker_color=eval_bar_colors,
    text=eval_outcomes, textposition='inside', textfont_size=9,
    hovertemplate='%{y}<br>Outcome: %{text}<extra></extra>',
    showlegend=False
), row=2, col=1)
fig.update_xaxes(visible=False, row=2, col=1)

# Bottom-right: SEC event log summary
from collections import Counter
event_types = Counter()
for e in sec_events:
    prefix = e['label'].split('_')[0] if '_' in e['label'] else e['label']
    event_types[f"{prefix} ({e.get('status','?')})"] += 1
top_events = event_types.most_common(10)
if top_events:
    fig.add_trace(go.Bar(
        y=[e[0] for e in top_events], x=[e[1] for e in top_events],
        orientation='h', marker_color='#8da0cb',
        hovertemplate='%{y}: %{x} events<extra></extra>',
        showlegend=False
    ), row=2, col=2)
fig.update_xaxes(title_text='Event Count', row=2, col=2)

# Legend annotations
for status, color in status_colors.items():
    fig.add_trace(go.Bar(x=[None], y=[None], marker_color=color, name=status, showlegend=True))

fig.update_layout(
    height=700, template='plotly_white',
    title_text='FinView Agent Evaluation Dashboard',
    legend=dict(orientation='h', yanchor='bottom', y=-0.12, xanchor='center', x=0.5)
)
fig.show()